In [24]:
import pandas as pd

# 讀取合併後的檔案
file_path = "D:/llm/select_de_210.csv"
# file_path ="C:/Users/user/Desktop/llm/merged_posts410_cut.csv"
df = pd.read_csv(file_path)

# 確保欄位正確
df['word'] = df['word'].fillna('').astype(str)
df['caption'] = df['caption'].fillna('').astype(str)

# 定義 A/W/U 的標籤集合
target_labels = {
    "D": ["S-D", "B-D", "M-D", "E-D"],
    "W": ["S-W", "B-W", "M-W", "E-W"],
    "A": ["S-A", "B-A", "M-A", "E-A"],
    "U": ["S-U", "B-U", "M-U", "E-U"]
}

def check_emo_in_post(labels, emo):
    """檢查單篇是否包含某情緒類別"""
    # 單詞
    if any(lbl == f"S-{emo}" for lbl in labels):
        return True
    # 序列 B-M-E
    i = 0
    while i < len(labels):
        if labels[i] == f"B-{emo}":
            j = i + 1
            # 跳過任意數量的 M-*
            while j < len(labels) and labels[j] == f"M-{emo}":
                j += 1
            # 檢查是否有 E-*
            if j < len(labels) and labels[j] == f"E-{emo}":
                return True
            i = j  # 跳到下一個可能的 B-*
        else:
            i += 1
    return False

# 主流程
valid_posts = []
stats = {"D":0, "A": 0, "W": 0, "U": 0}
combo_stats = {}

for post_id, group in df.groupby("post_id"):
    labels = group['human_label'].tolist()
    has_flags = {emo: check_emo_in_post(labels, emo) for emo in ["D", "W", "A", "U"]}

    if any(has_flags.values()):  # 至少含一種
        caption = group["caption"].iloc[0]
        valid_posts.append({"post_id": post_id, "caption": caption})

        # 單類別統計（可重複）
        for emo, flag in has_flags.items():
            if flag:
                stats[emo] += 1

        # 多類別組合統計（互斥）
        combo_key = "+".join([emo for emo, flag in has_flags.items() if flag])
        combo_stats[combo_key] = combo_stats.get(combo_key, 0) + 1

# 轉成 DataFrame
df_posts = pd.DataFrame(valid_posts)

print("符合條件的貼文數：", len(df_posts))

print("\n📊 類別統計（可重複）：")
for emo, count in stats.items():
    print(f"{emo}: {count} 篇")

print("\n📊 類別組合統計（互斥）：")
for combo, count in combo_stats.items():
    print(f"{combo}: {count} 篇")

# 輸出
OUT_CSV = "D:/llm/original_posts_awu.csv"
df_posts.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

print("\n✅ 完成，已輸出：", OUT_CSV)


符合條件的貼文數： 194

📊 類別統計（可重複）：
D: 147 篇
A: 19 篇
W: 77 篇
U: 57 篇

📊 類別組合統計（互斥）：
D: 66 篇
U: 16 篇
D+U: 22 篇
D+W: 35 篇
W: 19 篇
D+W+A: 4 篇
D+W+U: 12 篇
W+U: 5 篇
D+A: 6 篇
D+A+U: 2 篇
A: 5 篇
W+A: 2 篇

✅ 完成，已輸出： D:/llm/original_posts_awu.csv


In [25]:
INPUT_CSV = "D:/llm/original_posts_awu.csv"
# 假設欄位 ["post_id", "text"]
records_df = pd.read_csv(INPUT_CSV)[["post_id", "caption"]].dropna().reset_index(drop=True)

print("載入樣本數：", len(records_df))
print(records_df.head())

載入樣本數： 194
   post_id                                            caption
0     1001  人們總是說「你還沒有體驗過⋯⋯」、「你還沒有等到⋯⋯」，希望你能再多活一點。\n可是他們不知...
1     1002                 然後就睡著了。\n\n睡吧，睡吧，夢裡什麼都有，唯一沒有的，是明天。
2     1003  「請問他是您的⋯⋯？目前是在家中還是醫院呢？」電話那端傳來小心翼翼的試探，聲音沉穩、讓人放鬆...
3     1004  國小的時候老師說：「你們知道嗎，平均每3.9秒就有一個人餓死。」\n他的原意是要我們珍惜食物...
4     1005  最近都是註定不眠的夜。\n我不斷想起幾個句子：「我緊緊抱你的時候這世界好多人死」（阿芒）、「...


In [26]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 2080 Ti


In [13]:
import torch
print("CUDA 是否可用:", torch.cuda.is_available())

CUDA 是否可用: True


## 四步驟分開執行


### 🔹1. stage1a_synopsis.py（生成 synopsis

In [20]:
!pip install huggingface_hub==0.23.0
from huggingface_hub import login

  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.1.7
    Uninstalling huggingface_hub-1.1.7:
      Successfully uninstalled huggingface_hub-1.1.7


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.1.1 requires huggingface-hub>=0.24.0, but you have huggingface-hub 0.23.0 which is incompatible.
transformers 4.47.1 requires huggingface-hub<1.0,>=0.24.0, but you have huggingface-hub 0.23.0 which is incompatible.


In [21]:
login()


In [ ]:
import pandas as pd
import json, re
from transformers import pipeline
from tqdm.auto import tqdm

# ========= 載入資料 =========
df = pd.read_csv("D:/llm/original_posts_awu.csv")
df['caption'] = df['caption'].fillna('').astype(str)

print("載入樣本數：", len(df))
print(df.head())
# ========= 建立模型 =========
generator = pipeline("text-generation", model="meta-llama/Llama-3.2-3B-Instruct")

# ========= JSON parsing 函式 =========
def clean_json_output(text):
    candidates = re.findall(r"\{.*?\}", text, flags=re.S)
    for cand in candidates[::-1]:
        try:
            parsed = json.loads(cand)
            if "synopsis" in parsed:
                return parsed
        except:
            continue
    return None

# ========= 生成 synopsis (加 retry) =========
def generate_synopsis(post, max_retries=5):
    prompt = f"""
你是一位心理健康助理，請根據以下內容生成**簡短摘要**。

貼文：{post}

請務必遵守：
1. 輸出必須是 **繁體中文**。
2. 摘要長度需在 30–80 個字之間。
3. 僅能輸出 JSON 格式，內容放在 <OUTPUT> 和 </OUTPUT> 標籤內。
4. 不允許任何解釋或額外文字。
5. 重點捕捉發文者的情境、困難或心理症狀（例如：自我否定、失去興趣、無力感）。

範例輸出：
<OUTPUT>
{{"synopsis": "覺得每天都很疲憊且感到無力，對生活失去了熱情。""}}
</OUTPUT>
"""
    out_text = ""
    for attempt in range(max_retries):
        out_text = generator(prompt, max_new_tokens=200, temperature=0.5, return_full_text=False)[0]["generated_text"]

        # 嘗試抓 <OUTPUT>
        m = re.search(r"<OUTPUT>(.*?)</OUTPUT>", out_text, flags=re.S)
        if m:
            try:
                return json.loads(m.group(1).strip()).get("synopsis", "")
            except:
                pass

        # 嘗試多組 {…}
        parsed = clean_json_output(out_text)
        if parsed:
            return parsed.get("synopsis", "")

    # 若所有嘗試都失敗
    return "【解析失敗】" + out_text.strip()

# ========= 主程式 =========
results = []
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Stage1a - Synopsis"):
    post = row['caption']
    if not post.strip():
        continue
    synopsis = generate_synopsis(post)
    results.append({"post_id": idx, "original_post": post, "synopsis": synopsis})

# ========= 輸出 =========
out_path = "D:/llm/synthetic_3stage/stage1a_synopsis.csv"
pd.DataFrame(results).to_csv(out_path, index=False, encoding="utf-8-sig")

print(f"完成！Stage1a 輸出：{len(results)} 筆，已存檔：{out_path}")


載入樣本數： 194
   post_id                                            caption
0     1001  人們總是說「你還沒有體驗過⋯⋯」、「你還沒有等到⋯⋯」，希望你能再多活一點。\n可是他們不知...
1     1002                 然後就睡著了。\n\n睡吧，睡吧，夢裡什麼都有，唯一沒有的，是明天。
2     1003  「請問他是您的⋯⋯？目前是在家中還是醫院呢？」電話那端傳來小心翼翼的試探，聲音沉穩、讓人放鬆...
3     1004  國小的時候老師說：「你們知道嗎，平均每3.9秒就有一個人餓死。」\n他的原意是要我們珍惜食物...
4     1005  最近都是註定不眠的夜。\n我不斷想起幾個句子：「我緊緊抱你的時候這世界好多人死」（阿芒）、「...


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model-00001-of-00002.safetensors:  25%|##4       | 1.24G/4.97G [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Device set to use cuda:0


Stage1a - Synopsis:   0%|          | 0/194 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_to

完成！Stage1a 輸出：194 筆，已存檔：D:/llm/synthetic_3stage/stage1a_synopsis.csv


In [ ]:
# import pandas as pd
# import json, re
# from transformers import pipeline
# from tqdm.auto import tqdm
# from transformers import AutoModelForCausalLM, AutoTokenizer
# # ========= 載入資料 =========
# df = pd.read_csv("D:/llm/original_posts_awu.csv")
# df['caption'] = df['caption'].fillna('').astype(str)

# print("載入樣本數：", len(df))
# print(df.head())
# # ========= 建立模型 =========
# generator = pipeline("text-generation", model="meta-llama/Llama-3.2-3B-Instruct")


# # ========= JSON parsing 函式 =========
# def clean_json_output(text):
#     candidates = re.findall(r"\{.*?\}", text, flags=re.S)
#     for cand in candidates[::-1]:
#         try:
#             parsed = json.loads(cand)
#             if "synopsis" in parsed:
#                 return parsed
#         except:
#             continue
#     return None

# # ========= 生成 synopsis (加 retry) =========
# def generate_synopsis(post, max_retries=3):
#     prompt = f"""
# 你是一位心理健康助理，負責根據下列貼文內容生成一段「摘要」。

# 【貼文內容】：
# {post}

# 【指令】：
# 1. 僅能使用貼文內容中的資訊，以流暢的繁體中文撰寫一段「客觀摘要」。
# 2. 摘要必須客觀，需捕捉發文者的主要事件、困境、情境，但不能加入任何情緒判斷。
# 3. 不得重複貼文中的原句，必須以新的語句重新敘述內容。
# 4. 不得使用第一人稱（不得出現「我、我們」），須以第三人稱描述，如「該使用者」。
# 5. 不得加入任何額外說明、推論或外部資訊。不能跨題、不能引用其他提示。
# 6. 僅能輸出 **單行 JSON**，不得含有空白、換行、標記符號、特殊字元。
# 7. JSON 結構必須為：{{"synopsis":"內容"}}

# 請生成輸出：
# """

#     out_text = ""
#     for attempt in range(max_retries):
#         out_text = generator(prompt, max_new_tokens=150, temperature=0.5, return_full_text=False)[0]["generated_text"]

#         # 嘗試抓 <OUTPUT>
#         m = re.search(r"<OUTPUT>(.*?)</OUTPUT>", out_text, flags=re.S)
#         if m:
#             try:
#                 return json.loads(m.group(1).strip()).get("synopsis", "")
#             except:
#                 pass

#         # 嘗試多組 {…}
#         parsed = clean_json_output(out_text)
#         if parsed:
#             return parsed.get("synopsis", "")

#     # 若所有嘗試都失敗
#     return "【解析失敗】" + out_text.strip()

# # ========= 主程式 =========
# results = []
# for idx, row in tqdm(df.iterrows(), total=len(df), desc="Stage1a - Synopsis"):
#     post = row['caption']
#     if not post.strip():
#         continue
#     synopsis = generate_synopsis(post)
#     results.append({"post_id": idx, "original_post": post, "synopsis": synopsis})

# # ========= 輸出 =========
# out_path = "D:/llm/synthetic_3stage_210/stage1a_synopsis.csv"
# pd.DataFrame(results).to_csv(out_path, index=False, encoding="utf-8-sig")

# print(f"完成！Stage1a 輸出：{len(results)} 筆，已存檔：{out_path}")


載入樣本數： 5
   post_id                                            caption
0     1001  人們總是說「你還沒有體驗過⋯⋯」、「你還沒有等到⋯⋯」，希望你能再多活一點。\n可是他們不知...
1     1002                 然後就睡著了。\n\n睡吧，睡吧，夢裡什麼都有，唯一沒有的，是明天。
2     1003  「請問他是您的⋯⋯？目前是在家中還是醫院呢？」電話那端傳來小心翼翼的試探，聲音沉穩、讓人放鬆...
3     1004  國小的時候老師說：「你們知道嗎，平均每3.9秒就有一個人餓死。」\n他的原意是要我們珍惜食物...
4     1005  最近都是註定不眠的夜。\n我不斷想起幾個句子：「我緊緊抱你的時候這世界好多人死」（阿芒）、「...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


Stage1a - Synopsis:   0%|          | 0/5 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


完成！Stage1a 輸出：5 筆，已存檔：D:/llm/synthetic_3stage_210/stage1a_synopsis.csv


In [ ]:
# import pandas as pd
# import json, re
# from transformers import pipeline
# from tqdm.auto import tqdm
# from transformers import AutoModelForCausalLM, AutoTokenizer
# # ========= 載入資料 =========
# df = pd.read_csv("D:/llm/original_posts_awu.csv")
# df['caption'] = df['caption'].fillna('').astype(str)
# df = df.head(3)
# print("載入樣本數：", len(df))
# print(df.head())

# # ========= 建立模型 =========
# generator = pipeline("text-generation", model="meta-llama/Llama-3.2-3B-Instruct")
# # model_name = "meta-llama/Llama-3.2-3B-Instruct"

# # tokenizer = AutoTokenizer.from_pretrained(model_name)
# # model = AutoModelForCausalLM.from_pretrained(
# #     model_name,
# #     load_in_4bit=True,
# #     device_map="auto"
# # )

# # generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

# # ========= JSON parsing 函式 =========
# def clean_json_output(text):
#     """
#     從 LLM 輸出中抓出第一個合法的 JSON，並回傳 dict
#     """
#     candidates = re.findall(r"\{.*?\}", text, flags=re.S)
#     for cand in candidates[::-1]:
#         try:
#             parsed = json.loads(cand)
#             if "synopsis" in parsed:
#                 return parsed
#         except:
#             continue
#     return None

# # ========= 生成 synopsis (加 retry) =========
# def generate_synopsis(post, max_retries=3):
#     prompt = f"""
# 你是一位心理健康助理，負責根據下列貼文內容生成一段「摘要」。

# 【貼文內容】：
# {post}

# 【指令】：
# 1. 僅能使用貼文內容中的資訊，以流暢的繁體中文撰寫一段「客觀摘要」。
# 2. 摘要必須客觀，需捕捉發文者的主要事件、困境、情境，但不能加入任何情緒判斷。
# 3. 不得重複貼文中的原句，必須以新的語句重新敘述內容。
# 4. 不得使用第一人稱（不得出現「我、我們」），須以第三人稱描述，如「該使用者」、「他」、「她」。
# 5. 不得加入任何額外說明、推論或外部資訊。不能跨題、不能引用其他提示。
# 6. 僅能輸出 **單行 JSON**，不得含有空白、換行、標記符號、特殊字元。
# 7. JSON 結構必須為：{{"synopsis":"內容"}}

# 請直接輸出這一行 JSON，勿輸出其他任何文字：
# """
#     out_text = ""
#     for attempt in range(max_retries):
#         out = generator(
#             prompt,
#             max_new_tokens=150,
#             temperature=0.5,
#             return_full_text=False
#         )[0]["generated_text"]

#         out_text = out.strip()

#         # 直接嘗試解析 JSON
#         parsed = clean_json_output(out_text)
#         if parsed:
#             return parsed.get("synopsis", "").strip()

#     # 若所有嘗試都失敗
#     return "【解析失敗】" + out_text.replace("\n", " ")

# # ========= 主程式：在原 df 上加 synopsis 欄位 =========
# df["synopsis"] = ""

# for idx, row in tqdm(df.iterrows(), total=len(df), desc="Stage1a - Synopsis"):
#     post = row["caption"]
#     if not post.strip():
#         continue
#     synopsis = generate_synopsis(post)
#     df.at[idx, "synopsis"] = synopsis

# # ========= 輸出 =========
# out_path = "D:/llm/synthetic_3stage_210/stage1a_synopsis.csv"
# df.to_csv(out_path, index=False, encoding="utf-8-sig")

# print(f"完成！Stage1a 輸出：{len(df)} 筆，已存檔：{out_path}")


載入樣本數： 3
   post_id                                            caption
0     1001  人們總是說「你還沒有體驗過⋯⋯」、「你還沒有等到⋯⋯」，希望你能再多活一點。\n可是他們不知...
1     1002                 然後就睡著了。\n\n睡吧，睡吧，夢裡什麼都有，唯一沒有的，是明天。
2     1003  「請問他是您的⋯⋯？目前是在家中還是醫院呢？」電話那端傳來小心翼翼的試探，聲音沉穩、讓人放鬆...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


Stage1a - Synopsis:   0%|          | 0/3 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


完成！Stage1a 輸出：3 筆，已存檔：D:/llm/synthetic_3stage_210/stage1a_synopsis.csv


### 🔹2. stage1b_sentiment.py（生成 sentiment）

In [ ]:
import pandas as pd
import json, re
from transformers import pipeline
from tqdm.auto import tqdm

# ========= 載入資料 =========
df = pd.read_csv("D:/llm/original_posts_awu.csv")
df['caption'] = df['caption'].fillna('').astype(str)

print("載入樣本數：", len(df))
print(df.head())
# ========= 建立模型 =========
generator = pipeline("text-generation", model="meta-llama/Llama-3.2-3B-Instruct")

# ========= JSON parsing 函式 =========
def clean_json_output(text):
    candidates = re.findall(r"\{.*?\}", text, flags=re.S)
    for cand in candidates[::-1]:
        try:
            parsed = json.loads(cand)
            if "sentiment" in parsed:
                return parsed
        except:
            continue
    return None

# ========= 生成 sentiment (加 retry) =========
def generate_sentiment(post, max_retries=3):
    prompt = f"""
你是一位心理健康助理，請根據以下貼文內容，生成一個**情緒分析**。

貼文：{post}

請務必遵守：
1. 分析貼文中反映出的**情緒狀態與強度**（例如：強烈的自責、持續的絕望、輕度焦慮）。
2. 輸出必須是 **繁體中文**。
3. 內容長度需在 20–60 個字之間。
4. 僅能輸出 JSON 格式，內容放在 <OUTPUT> 和 </OUTPUT> 標籤內。
5. 不允許任何解釋或額外文字。

範例輸出：
<OUTPUT>
{{"sentiment": "持續的無助感和情緒低落，符合中度憂鬱"}}
</OUTPUT>
"""
    out_text = ""
    for attempt in range(max_retries):
        out_text = generator(prompt, max_new_tokens=200, temperature=0.5, return_full_text=False)[0]["generated_text"]

        # 嘗試抓 <OUTPUT>
        m = re.search(r"<OUTPUT>(.*?)</OUTPUT>", out_text, flags=re.S)
        if m:
            try:
                return json.loads(m.group(1).strip()).get("sentiment", "")
            except:
                pass

        # 嘗試多組 {…}
        parsed = clean_json_output(out_text)
        if parsed:
            return parsed.get("sentiment", "")

    # 若所有嘗試都失敗
    return "【解析失敗】" + out_text.strip()

# ========= 主程式 =========
results = []
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Stage1b - Sentiment"):
    post = row['caption']
    if not post.strip():
        continue
    sentiment = generate_sentiment(post)
    results.append({"post_id": idx, "original_post": post, "sentiment": sentiment})

# ========= 輸出 =========
out_path = "D:/llm/synthetic_3stage/stage1b_sentiment.csv"
pd.DataFrame(results).to_csv(out_path, index=False, encoding="utf-8-sig")

print(f"完成！Stage1b 輸出：{len(results)} 筆，已存檔：{out_path}")


載入樣本數： 194
   post_id                                            caption
0     1001  人們總是說「你還沒有體驗過⋯⋯」、「你還沒有等到⋯⋯」，希望你能再多活一點。\n可是他們不知...
1     1002                 然後就睡著了。\n\n睡吧，睡吧，夢裡什麼都有，唯一沒有的，是明天。
2     1003  「請問他是您的⋯⋯？目前是在家中還是醫院呢？」電話那端傳來小心翼翼的試探，聲音沉穩、讓人放鬆...
3     1004  國小的時候老師說：「你們知道嗎，平均每3.9秒就有一個人餓死。」\n他的原意是要我們珍惜食物...
4     1005  最近都是註定不眠的夜。\n我不斷想起幾個句子：「我緊緊抱你的時候這世界好多人死」（阿芒）、「...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


Stage1b - Sentiment:   0%|          | 0/194 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

完成！Stage1b 輸出：194 筆，已存檔：D:/llm/synthetic_3stage/stage1b_sentiment.csv


In [ ]:
# import pandas as pd
# import json, re
# from transformers import pipeline
# from tqdm.auto import tqdm

# # ========= 載入資料 =========
# df = pd.read_csv("D:/llm/original_posts_awu.csv")
# df['caption'] = df['caption'].fillna('').astype(str)
# df = df.head(5)
# print("載入樣本數：", len(df))
# print(df.head())
# # ========= 建立模型 =========
# generator = pipeline("text-generation", model="meta-llama/Llama-3.2-3B-Instruct")

# # ========= JSON parsing 函式 =========
# def clean_json_output(text):
#     candidates = re.findall(r"\{.*?\}", text, flags=re.S)
#     for cand in candidates[::-1]:
#         try:
#             parsed = json.loads(cand)
#             if "sentiment" in parsed:
#                 return parsed
#         except:
#             continue
#     return None

# # ========= 生成 sentiment (加 retry) =========
# def generate_sentiment(post, max_retries=3):
#     prompt = f"""
# 你是一位心理健康助理，負責根據下列貼文內容生成一段「情緒分析」。

# 【貼文內容】：
# {post}

# 【指令】：
# 1. 僅能使用貼文內容中的資訊，以流暢的繁體中文撰寫一份的「情緒分析」。
# 2. 識別並詳述貼文反映出的「具體情緒」與「情緒強度」。
# 3. 不得直接重複貼文原文句子，需以不同句法重新表述。
# 4. 不得使用第一人稱（不得出現「我、我們」），須以第三人稱描述，如「該使用者」、「他」或「她」。
# 5. 不得包含任何額外說明，不可切換任務，不可加入外部資訊。
# 6. 僅能輸出 **單行 JSON**，不得含任何空白、換行、特殊字元。
# 7. JSON 結構必須為：{{"sentiment": "內容"}}。

# 請生成輸出：
# """
#     out_text = ""
#     for attempt in range(max_retries):
#         out_text = generator(prompt, max_new_tokens=200, temperature=0.5, return_full_text=False)[0]["generated_text"]

#         # 嘗試抓 <OUTPUT>
#         m = re.search(r"<OUTPUT>(.*?)</OUTPUT>", out_text, flags=re.S)
#         if m:
#             try:
#                 return json.loads(m.group(1).strip()).get("sentiment", "")
#             except:
#                 pass

#         # 嘗試多組 {…}
#         parsed = clean_json_output(out_text)
#         if parsed:
#             return parsed.get("sentiment", "")

#     # 若所有嘗試都失敗
#     return "【解析失敗】" + out_text.strip()

# # ========= 主程式 =========
# results = []
# for idx, row in tqdm(df.iterrows(), total=len(df), desc="Stage1b - Sentiment"):
#     post = row['caption']
#     if not post.strip():
#         continue
#     sentiment = generate_sentiment(post)
#     results.append({"post_id": idx, "original_post": post, "sentiment": sentiment})

# # ========= 輸出 =========
# out_path = "D:/llm/synthetic_3stage_210/stage1b_sentiment.csv"
# pd.DataFrame(results).to_csv(out_path, index=False, encoding="utf-8-sig")

# print(f"完成！Stage1b 輸出：{len(results)} 筆，已存檔：{out_path}")


載入樣本數： 5
   post_id                                            caption
0     1001  人們總是說「你還沒有體驗過⋯⋯」、「你還沒有等到⋯⋯」，希望你能再多活一點。\n可是他們不知...
1     1002                 然後就睡著了。\n\n睡吧，睡吧，夢裡什麼都有，唯一沒有的，是明天。
2     1003  「請問他是您的⋯⋯？目前是在家中還是醫院呢？」電話那端傳來小心翼翼的試探，聲音沉穩、讓人放鬆...
3     1004  國小的時候老師說：「你們知道嗎，平均每3.9秒就有一個人餓死。」\n他的原意是要我們珍惜食物...
4     1005  最近都是註定不眠的夜。\n我不斷想起幾個句子：「我緊緊抱你的時候這世界好多人死」（阿芒）、「...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


Stage1b - Sentiment:   0%|          | 0/5 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


完成！Stage1b 輸出：5 筆，已存檔：D:/llm/synthetic_3stage_210/stage1b_sentiment.csv


### 🔹3. stage2a_synthetic_synopsis.py（生成 synthetic synopsis）

In [ ]:
# # generate_phq_mapping.py
# import pandas as pd
# import random

# # 讀 stage1a 或 stage1b 的任一檔，只要有 post_id 列即可
# df = pd.read_csv("C:/Users/user/Desktop/llm/stage1a_synopsis.csv")
# post_ids = df['post_id'].tolist()

# rows = []
# for pid in post_ids:
#     # 若要去掉 0 分：用 range(1,25)
#     # 若論文包含 0，可改成 range(0,25)
#     scores = random.sample(range(1,25), 3)  # 三個不同分數
#     for i, s in enumerate(scores):
#         rows.append({"post_id": int(pid), "phq8_score": int(s), "variant_idx": i+1})

# mapping_df = pd.DataFrame(rows)
# out_path = "C:/Users/user/Desktop/llm/post_phq_mapping.csv"
# mapping_df.to_csv(out_path, index=False, encoding="utf-8-sig")
# print(f"已產生 mapping：{out_path}，總行數：{len(mapping_df)}")


FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/user/Desktop/llm/stage1a_synopsis.csv'

In [29]:
import pandas as pd
import json, re, random
from transformers import pipeline
from tqdm.auto import tqdm

# ========= 載入 Stage1a 輸出 =========
df = pd.read_csv("D:/llm/synthetic_3stage/stage1a_synopsis.csv")
df['synopsis'] = df['synopsis'].fillna('').astype(str)
print("載入樣本數：", len(df))
print(df.head())
# ========= 建立模型 =========
generator = pipeline("text-generation", model="meta-llama/Llama-3.2-3B-Instruct")

# ========= JSON parsing 函式 =========
def clean_json_output(text):
    candidates = re.findall(r"\{.*?\}", text, flags=re.S)
    for cand in candidates[::-1]:
        try:
            parsed = json.loads(cand)
            if "synthetic_synopsis" in parsed:
                return parsed
        except:
            continue
    return None

# ========= Depression Level 映射 =========
DEP_LEVELS = {
    # range(1,5): "無憂鬱症狀",
    # range(5,10): "輕度憂鬱",
    range(10,15): "中度憂鬱",
    range(15,20): "中重度憂鬱",
    range(20,25): "重度憂鬱"
}
def get_dep_desc(score):
    for rng, desc in DEP_LEVELS.items():
        if score in rng:
            return desc
    return "未知"

# ========= 生成 synthetic synopsis (加強差異化) =========
def generate_synthetic_synopsis(original_synopsis, phq8_score, dep_desc, max_retries=3):
    prompt = f"""
你是一位心理健康助理，請根據以下資訊生成**新的摘要**。

原始摘要：{original_synopsis}
PHQ-8 分數：{phq8_score}
對應憂鬱程度：{dep_desc}

請務必遵守：
1. 生成一個新的摘要（30–80 個字）。
2. 摘要必須用 **繁體中文** 撰寫。
3. 新的摘要要與原始摘要不同，並且反映「{dep_desc}」的特徵。
4. 若 PHQ-8 分數不同，輸出的摘要必須有明顯差異。
5. 僅能輸出 JSON 格式，內容放在 <OUTPUT> 和 </OUTPUT> 標籤內。
6. 不允許任何解釋或額外文字。

範例輸出：
<OUTPUT>
{{"synthetic_synopsis": "作者在日常生活中出現輕微失落與缺乏動力"}}
</OUTPUT>
"""
    out_text = ""
    for attempt in range(max_retries):
        out_text = generator(prompt, max_new_tokens=200, temperature=0.7, return_full_text=False)[0]["generated_text"]

        # 嘗試抓 <OUTPUT>
        m = re.search(r"<OUTPUT>(.*?)</OUTPUT>", out_text, flags=re.S)
        if m:
            try:
                return json.loads(m.group(1).strip()).get("synthetic_synopsis", "")
            except:
                pass

        # 嘗試多組 {…}
        parsed = clean_json_output(out_text)
        if parsed:
            return parsed.get("synthetic_synopsis", "")

    return "【解析失敗】" + out_text.strip()

# ========= 主程式 =========
results = []
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Stage2a - Synthetic Synopsis"):
    synopsis = row['synopsis']
    if synopsis.startswith("【解析失敗】") or not synopsis.strip():
        continue

    # 每篇生成 3 個不同 PHQ-8 版本
    scores = random.sample(range(1, 25), 3)
    for score in scores:
        dep_desc = get_dep_desc(score)
        synthetic_synopsis = generate_synthetic_synopsis(synopsis, score, dep_desc)
        results.append({
            "post_id": row['post_id'],
            "original_synopsis": synopsis,
            "phq8_score": score,
            "dep_desc": dep_desc,
            "synthetic_synopsis": synthetic_synopsis
        })

# ========= 輸出 =========
out_path = "D:/llm/synthetic_3stage/stage2a_synthetic_synopsis.csv"
pd.DataFrame(results).to_csv(out_path, index=False, encoding="utf-8-sig")

print(f"完成！Stage2a 輸出：{len(results)} 筆，已存檔：{out_path}")


載入樣本數： 194
   post_id                                      original_post  \
0        0  人們總是說「你還沒有體驗過⋯⋯」、「你還沒有等到⋯⋯」，希望你能再多活一點。\n可是他們不知...   
1        1                 然後就睡著了。\n\n睡吧，睡吧，夢裡什麼都有，唯一沒有的，是明天。   
2        2  「請問他是您的⋯⋯？目前是在家中還是醫院呢？」電話那端傳來小心翼翼的試探，聲音沉穩、讓人放鬆...   
3        3  國小的時候老師說：「你們知道嗎，平均每3.9秒就有一個人餓死。」\n他的原意是要我們珍惜食物...   
4        4  最近都是註定不眠的夜。\n我不斷想起幾個句子：「我緊緊抱你的時候這世界好多人死」（阿芒）、「...   

                                            synopsis  
0                        感到自己無法控制情緒，感到無力且感到自己無法克服困難。  
1                          感覺自己無法應對生活中的壓力，感到焦慮和失去自信。  
2  【解析失敗】請輸出簡短摘要，注意遵守規則。 \n\n**簡短摘要：**\n我是一位心理健康助...  
3                                感到孤單和無助，對未來的前景感到擔憂。  
4                              感到被動、無力，無法改變自己、自己的生活。  


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


Stage2a - Synthetic Synopsis:   0%|          | 0/194 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


KeyboardInterrupt: 

In [ ]:
# ========= 統計表 =========
results = pd.read_csv("D:/llm/synthetic_3stage/stage2a_synthetic_synopsis.csv")
df_results = pd.DataFrame(results)
print("載入樣本數：", len(df_results))
if not df_results.empty:
    stats = df_results.groupby("dep_desc").size().reset_index(name="count")
    print("\nPHQ-8 區間分佈統計：")
    print(stats.to_string(index=False))
else:
    print("⚠️ 沒有成功生成任何結果")


載入樣本數： 351

PHQ-8 區間分佈統計：
dep_desc  count
    中度憂鬱     75
   中重度憂鬱     79
   無憂鬱症狀     59
    輕度憂鬱     66
    重度憂鬱     72


### 🔹4. stage2b_synthetic_sentiment.py（生成 synthetic sentiment）

In [ ]:
import pandas as pd
import json, re
from transformers import pipeline
from tqdm.auto import tqdm

# ========= 載入 Stage1b (原始情緒) 與 Stage2a (對應分數) =========
df_sent = pd.read_csv("D:/llm/synthetic_3stage/stage1b_sentiment.csv")
df_stage2a = pd.read_csv("D:/llm/synthetic_3stage/stage2a_synthetic_synopsis.csv")

# 確保文字欄位
df_sent['sentiment'] = df_sent['sentiment'].fillna('').astype(str)

# 做查詢表 (post_id -> 原始 sentiment)
sent_map = {int(r["post_id"]): r["sentiment"] for r in df_sent.to_dict("records")}

# ========= 建立模型 =========
generator = pipeline("text-generation", model="meta-llama/Llama-3.2-3B-Instruct")

# ========= JSON parsing =========
def clean_json_output(text):
    candidates = re.findall(r"\{.*?\}", text, flags=re.S)
    for cand in candidates[::-1]:
        try:
            parsed = json.loads(cand)
            if "synthetic_sentiment" in parsed:
                return parsed
        except:
            continue
    return None

# ========= 生成 synthetic sentiment =========
def generate_synthetic_sentiment(original_sentiment, phq8_score, dep_desc, max_retries=3):
    prompt = f"""
你是一位心理健康助理，請根據以下資訊生成**新的情緒分析**。

原始情緒分析：{original_sentiment}
PHQ-8 分數：{phq8_score}
對應憂鬱程度：{dep_desc}

請務必遵守：
1. 生成一個新的情緒分析（30–80 個字）。
2. 必須用 **繁體中文** 撰寫。
3. 新的情緒分析要與原始不同，並且反映「{dep_desc}」的特徵。
4. 同一篇若 PHQ-8 分數不同，輸出的情緒分析必須有明顯差異。
5. 僅能輸出 JSON 格式，內容放在 <OUTPUT> 和 </OUTPUT> 標籤內。
6. 不允許任何解釋或額外文字。

範例輸出：
<OUTPUT>
{{"synthetic_sentiment": "作者表現出輕度的焦慮與不安"}}
</OUTPUT>
"""
    out_text = ""
    for attempt in range(max_retries):
        out_text = generator(prompt, max_new_tokens=200, temperature=0.7, return_full_text=False)[0]["generated_text"]

        m = re.search(r"<OUTPUT>(.*?)</OUTPUT>", out_text, flags=re.S)
        if m:
            try:
                return json.loads(m.group(1).strip()).get("synthetic_sentiment", "")
            except:
                pass

        parsed = clean_json_output(out_text)
        if parsed:
            return parsed.get("synthetic_sentiment", "")

    return "【解析失敗】" + out_text.strip()

# ========= 主程式 =========
results = []
for _, row in tqdm(df_stage2a.iterrows(), total=len(df_stage2a), desc="Stage2b - Synthetic Sentiment"):
    pid = int(row['post_id'])
    score = int(row['phq8_score'])
    dep_desc = row['dep_desc']

    original_sent = sent_map.get(pid, "")
    if not original_sent or original_sent.startswith("【解析失敗】"):
        continue

    synthetic_sent = generate_synthetic_sentiment(original_sent, score, dep_desc)
    results.append({
        "post_id": pid,
        "original_sentiment": original_sent,
        "phq8_score": score,
        "dep_desc": dep_desc,
        "synthetic_sentiment": synthetic_sent
    })

out_path = "D:/llm/synthetic_3stage/stage2b_synthetic_sentiment.csv"
pd.DataFrame(results).to_csv(out_path, index=False, encoding="utf-8-sig")

print(f"完成！Stage2b 輸出：{len(results)} 筆，已存檔：{out_path}")


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 30.97it/s]
Device set to use cuda:0
Stage2b - Synthetic Sentiment:   0%|          | 0/351 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Stage2b - Synthetic Sentiment:   3%|▎         | 12/351 [01:48<39:30,  6.99s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Stage2b - Synthetic Sentiment:   8%|▊         | 29/351 [04:03<34:40,  6.46s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Stage2b - Synthetic Sentiment:  25%|██▌       | 89/351 [11:36<27:35,  6.32s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pa

完成！Stage2b 輸出：351 筆，已存檔：C:/Users/user/Desktop/llm/synthetic_3stage/stage2b_synthetic_sentiment.csv


In [ ]:
# ========= 統計表 =========
results = pd.read_csv("D:/llm/synthetic_3stage/stage2b_synthetic_sentiment.csv")
df_results = pd.DataFrame(results)
print("合併後資料筆數：", len(df_results))
if not df_results.empty:
    stats = df_results.groupby("dep_desc").size().reset_index(name="count")
    print("\nPHQ-8 區間分佈統計：")
    print(stats.to_string(index=False))
else:
    print("⚠️ 沒有成功生成任何結果")


合併後資料筆數： 351

PHQ-8 區間分佈統計：
dep_desc  count
    中度憂鬱     75
   中重度憂鬱     79
   無憂鬱症狀     59
    輕度憂鬱     66
    重度憂鬱     72


Stage3 :
把 Stage2a + Stage2b 的結果，轉換成自然的 IG 貼文（caption）

In [ ]:
# # Stage3_generate_with_origid.py 保留原id的版本

# import re
# import pandas as pd
# from tqdm.auto import tqdm
# from transformers import pipeline
# from sentence_transformers import SentenceTransformer, util

# # ========= 路徑 =========
# STAGE2A_PATH = "D:/llm/synthetic_3stage/stage2a_synthetic_synopsis.csv"
# STAGE2B_PATH = "D:/llm/synthetic_3stage/stage2b_synthetic_sentiment.csv"
# OUT_PATH = "D:/llm/synthetic_3stage/stage3_synthetic_posts_new.csv"

# # ========= 載入資料 =========
# df_syn = pd.read_csv(STAGE2A_PATH)
# df_sen = pd.read_csv(STAGE2B_PATH)
# df = pd.merge(df_syn, df_sen, on=["post_id", "phq8_score", "dep_desc"], how="inner")

# print("合併後資料筆數：", len(df))

# # ========= 模型 =========
# GEN_MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
# generator = pipeline("text-generation", model=GEN_MODEL_NAME,
#                      device_map="auto", torch_dtype="auto")

# embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# # ========= 清理函式 =========
# def cleanup_generated_text(text: str) -> str:
#     if not isinstance(text, str):
#         return ""
#     s = re.sub(r"```[\s\S]*?```", " ", text)   # code block
#     s = re.sub(r"\{[\s\S]*?\}", " ", s)        # JSON
#     s = re.sub(r"<[^>]+>", " ", s)             # HTML
#     s = re.sub(r"[A-Za-z]", "", s)             # 英文
#     s = re.sub(r"[^\u4e00-\u9fff0-9，。！？：；、（）「」『』．\s]", " ", s)  # 僅保留中文+數字
#     s = re.sub(r"\s+", " ", s).strip()
#     return s

# # ========= 驗證函式 =========
# def ngram_overlap(a, b, n=3):
#     a_ngrams = set([a[i:i+n] for i in range(max(0, len(a)-n+1))])
#     b_ngrams = set([b[i:i+n] for i in range(max(0, len(b)-n+1))])
#     if not a_ngrams or not b_ngrams: return 0.0
#     return len(a_ngrams & b_ngrams) / max(1, min(len(a_ngrams), len(b_ngrams)))

# def cosine_distance(a_emb, b_emb):
#     return 1 - util.cos_sim(a_emb, b_emb).item()

# def validate_caption(post, synopsis, sentiment,
#                      min_len=80, max_len=200,
#                      max_ngram_overlap=0.8, min_cos_dist=0.1):
#     if not post or not isinstance(post, str): return False, "empty"
#     L = len(post)
#     if not (min_len <= L <= max_len): return False, f"len={L}"
#     ov1 = ngram_overlap(post, synopsis)
#     ov2 = ngram_overlap(post, sentiment)
#     if ov1 > max_ngram_overlap: return False, f"ng_syn={ov1:.3f}"
#     if ov2 > max_ngram_overlap: return False, f"ng_sen={ov2:.3f}"
#     ref = (str(synopsis) + " " + str(sentiment)).strip()
#     if not ref: return False, "no_ref"
#     dist = cosine_distance(embedder.encode(post, convert_to_tensor=True),
#                            embedder.encode(ref, convert_to_tensor=True))
#     if dist < min_cos_dist: return False, f"cos={dist:.3f}"
#     return True, "ok"

# # ========= 生成函式 =========
# def generate_post(synopsis, sentiment, phq8_score, dep_desc):
#     prompt = f"""
# 你是一位社群使用者，請根據以下資訊生成**一則完整 IG 貼文**：
# - 摘要：{synopsis}
# - 情緒分析：{sentiment}
# - PHQ-8 分數：{phq8_score}（{dep_desc}）

# 請務必遵守：
# 1. 僅輸出一則繁體中文貼文（80–200字）。
# 2. 不得輸出 JSON、程式碼、英文、emoji。
# 3. 使用自然口語，像是在 IG 抒發心情。
# """
#     out = generator(prompt, max_new_tokens=300,
#                     do_sample=True, temperature=0.5, top_p=0.9,
#                     return_full_text=False)[0]["generated_text"]
#     return cleanup_generated_text(out)

# # ========= 主程式 =========
# results = []
# new_post_id = 801

# for _, row in tqdm(df.iterrows(), total=len(df), desc="Stage3"):
#     orig_post_id = int(row["post_id"])   # 🔑 保留 Stage2 的 post_id
#     syn, sen = str(row["synthetic_synopsis"]), str(row["synthetic_sentiment"])
#     phq, dep = int(row["phq8_score"]), row["dep_desc"]

#     if syn.startswith("【解析失敗】") or sen.startswith("【解析失敗】"):
#         results.append({
#             "post_id": new_post_id,
#             "orig_post_id": orig_post_id,   # 新增這欄位
#             "caption": "【解析失敗】 stage2資料缺失",
#             "phq8_score": phq, "dep_desc": dep,
#             "valid": False, "note": "stage2_fail"
#         })
#     else:
#         cap = generate_post(syn, sen, phq, dep)
#         ok, reason = validate_caption(cap, syn, sen)
#         results.append({
#             "post_id": new_post_id,
#             "orig_post_id": orig_post_id,   # 新增這欄位
#             "caption": cap,
#             "phq8_score": phq, "dep_desc": dep,
#             "valid": ok, "note": reason
#         })
#     new_post_id += 1

# # ========= 輸出 =========
# out_df = pd.DataFrame(results)
# out_df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

# print("Stage3 完成 ✅，已包含 orig_post_id")
# print("輸出檔案：", OUT_PATH)


合併後資料筆數： 351


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.36s/it]
Device set to use cuda:0
Stage3:  31%|███       | 108/351 [3:11:53<7:11:45, 106.61s/it] 


KeyboardInterrupt: 

In [ ]:
import re
import random
import pandas as pd
from tqdm.auto import tqdm
from transformers import pipeline
from sentence_transformers import SentenceTransformer, util

# ========= 路徑 =========
STAGE2A_PATH = "D:/llm/synthetic_3stage/stage2a_synthetic_synopsis.csv"
STAGE2B_PATH = "D:/llm/synthetic_3stage/stage2b_synthetic_sentiment.csv"
OUT_PATH = "D:/llm/synthetic_3stage/stage3_factors_posts.csv"
OUT_STATS = "D:/llm/synthetic_3stage/stage3_factors_stats.csv"

# ========= 載入資料 =========
df_syn = pd.read_csv(STAGE2A_PATH)
df_syn = df_syn.head(5)
df_sen = pd.read_csv(STAGE2B_PATH)
df_sen = df_sen.head(5)
df = pd.merge(df_syn, df_sen, on=["post_id", "phq8_score", "dep_desc"], how="inner")

print("合併後資料筆數：", len(df))

# ========= 模型 =========
GEN_MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
generator = pipeline("text-generation", model=GEN_MODEL_NAME,
                     device_map="auto", torch_dtype="auto")

embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# ========= Psychosocial factors (外在情境因子, Section 3.2) =========
PSYCHOSOCIAL_FACTORS = [
    "憂鬱", "焦慮", "無望感", "孤獨感",
    "霸凌", "家庭衝突", "感情挫折", "經濟壓力",
    "失業", "學業壓力", "歧視", "慢性疾病",
    "藥物濫用", "自殺意念"
]


# ========= 驗證函式 =========
def validate_caption(post, synopsis, sentiment,
                     min_len=20, max_len=300): 
    if not post or not isinstance(post, str): 
        return False, "empty"
    L = len(post)
    if not (min_len <= L <= max_len): 
        return False, f"len={L}"
    ref = (str(synopsis) + " " + str(sentiment)).strip()
    if not ref: 
        return False, "no_ref"
    return True, "ok"

# ========= 生成函式 =========
def cleanup_generated_text(text: str):
    # 移除多餘空白與奇怪符號
    text = re.sub(r"\n+", " ", text).strip()
    return text

# ========= Psychosocial factors (Section 3.2 全部 14 個) =========
PSYCHOSOCIAL_FACTORS = [
    "憂鬱", "焦慮", "無望感", "孤獨感",
    "霸凌", "家庭衝突", "感情挫折", "經濟壓力",
    "失業", "學業壓力", "歧視", "慢性疾病",
    "藥物濫用", "自殺意念"
]

# ========= 標籤與因子對應規則 =========
FACTOR_MAPPING = {
    "D": ["憂鬱", "焦慮", "孤獨感"],         # 抑鬱 → 內在情緒為主
    "A": ["學業壓力", "經濟壓力", "失業"],   # 快樂缺失 → 外在壓力
    "W": ["無望感", "感情挫折", "家庭衝突"], # 自責/罪惡 → 無望 + 關係
    "U": ["無望感", "自殺意念"],             # 自殺意念 → 關鍵因子
    "O": PSYCHOSOCIAL_FACTORS               # 無情緒 → 全部隨機
}

def choose_factor(sentiment_label: str):
    """根據 D/A/W/U/O 標籤挑選因子"""
    if sentiment_label in FACTOR_MAPPING:
        return random.choice(FACTOR_MAPPING[sentiment_label])
    else:
        return random.choice(PSYCHOSOCIAL_FACTORS)

# ========= 修改 generate_post =========
def generate_post(synopsis, sentiment, phq8_score, dep_desc):
    factor = random.choice(PSYCHOSOCIAL_FACTORS)  # 隨機選一個外在因子
    prompt = f"""
你是一位 IG 使用者，請根據以下資訊生成一則真實感的社群貼文：

- 摘要（核心情緒）：{synopsis}
- 情緒分析：{sentiment}
- PHQ-8 分數：{phq8_score}（{dep_desc}）
- 社會情境因子：{factor}

請務必遵守：
1. 僅輸出一則繁體中文 IG 貼文（大約 1–3 句，盡量精簡）。
2. 使用自然口語，非正式風格，可以使用 emoji 或斷句。
3. 不能輸出 JSON、程式碼、英文或解釋文字。
4. 貼文必須結合「核心情緒」與「社會情境因子」，模仿 IG 貼文的語氣。
"""
    out = generator(
        prompt,
        max_new_tokens=100,         # 論文使用的長度上限
        do_sample=True,             # 啟用隨機抽樣
        temperature=1.0,            # 論文指定
        top_p=0.9,                  # 論文指定
        return_full_text=False
    )[0]["generated_text"]
    return cleanup_generated_text(out)



# ========= 主程式 =========
results = []
stats = {"ok": 0, "fail_parse": 0, "fail_quality": 0}
new_post_id = 801

for _, row in tqdm(df.iterrows(), total=len(df), desc="Stage3"):
    orig_id = int(row["post_id"])   # 保留 Stage2 的 post_id
    syn, sen = str(row["synthetic_synopsis"]), str(row["synthetic_sentiment"])
    phq, dep = int(row["phq8_score"]), row["dep_desc"]

    if syn.startswith("【解析失敗】") or sen.startswith("【解析失敗】"):
        results.append({
            "orig_post_id": orig_id,
            "post_id": new_post_id,
            "caption": "【解析失敗】 stage2資料缺失",
            "phq8_score": phq, "dep_desc": dep,
            "valid": False, "note": "stage2_fail"
        })
        stats["fail_parse"] += 1
    else:
        cap = generate_post(syn, sen, phq, dep)
        ok, reason = validate_caption(cap, syn, sen)
        results.append({
            "orig_post_id": orig_id,
            "post_id": new_post_id,
            "caption": cap,
            "phq8_score": phq, "dep_desc": dep,
            "valid": ok, "note": reason
        })
        if ok: stats["ok"] += 1
        else: stats["fail_quality"] += 1
    new_post_id += 1

# ========= 輸出 =========
out_df = pd.DataFrame(results)
out_df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

stats_df = pd.DataFrame([
    {"type": "合格", "count": stats["ok"]},
    {"type": "解析失敗", "count": stats["fail_parse"]},
    {"type": "品質不合格", "count": stats["fail_quality"]},
    {"type": "總數", "count": len(results)}
])
stats_df["ratio"] = stats_df["count"] / len(results)
stats_df.to_csv(OUT_STATS, index=False, encoding="utf-8-sig")

print("Stage3 完成 ✅")
print(stats_df)


In [ ]:
import re
import random
import pandas as pd
from tqdm.auto import tqdm
from transformers import pipeline
from sentence_transformers import SentenceTransformer, util

# ========= 路徑 =========
STAGE2A_PATH = "D:/llm/synthetic_3stage/stage2a_synthetic_synopsis.csv"
STAGE2B_PATH = "D:/llm/synthetic_3stage/stage2b_synthetic_sentiment.csv"
OUT_PATH = "D:/llm/synthetic_3stage/stage3_factors_posts.csv"
OUT_STATS = "D:/llm/synthetic_3stage/stage3_factors_stats.csv"

# ========= 載入資料 =========
df_syn = pd.read_csv(STAGE2A_PATH)
df_syn = df_syn.head(5)
df_sen = pd.read_csv(STAGE2B_PATH)
df_sen = df_sen.head(5)
df = pd.merge(df_syn, df_sen, on=["post_id", "phq8_score", "dep_desc"], how="inner")

print("合併後資料筆數：", len(df))

# ========= 模型 =========
GEN_MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
generator = pipeline("text-generation", model=GEN_MODEL_NAME,
                     device_map="auto", torch_dtype="auto")

embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# ========= Psychosocial factors (外在情境因子, Section 3.2) =========
PSYCHOSOCIAL_FACTORS = [
    "憂鬱", "焦慮", "無望感", "孤獨感",
    "霸凌", "家庭衝突", "感情挫折", "經濟壓力",
    "失業", "學業壓力", "歧視", "慢性疾病",
    "藥物濫用", "自殺意念"
]


# ========= 驗證函式 =========
def validate_caption(post, synopsis, sentiment,
                     min_len=20, max_len=300): 
    if not post or not isinstance(post, str): 
        return False, "empty"
    L = len(post)
    if not (min_len <= L <= max_len): 
        return False, f"len={L}"
    ref = (str(synopsis) + " " + str(sentiment)).strip()
    if not ref: 
        return False, "no_ref"
    return True, "ok"

# ========= 生成函式 =========
def cleanup_generated_text(text: str):
    # 移除多餘空白與奇怪符號
    text = re.sub(r"\n+", " ", text).strip()
    return text
# def limit_emojis(text, max_emojis=3):
#     # 找出所有 emoji
#     emojis = re.findall(r"[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF]+", text)
#     if len(emojis) <= max_emojis:
#         return text
#     # 隨機保留最多 max_emojis 個
#     keep = set(random.sample(emojis, max_emojis))
#     return "".join(ch for ch in text if ch not in emojis or ch in keep)


# def cleanup_generated_text(text: str):
#     # 移除多餘空白與換行
#     text = re.sub(r"\s+", " ", text).strip()
    
#     # 限制 emoji 數量 (最多 3 個)
#     text = limit_emojis(text, max_emojis=2)
    
#     # # 偵測是否含有英文，若有則翻譯成繁體中文
#     # if re.search(r"[A-Za-z]", text):
#     #     try:
#     #         text = translator.translate(text, src="en", dest="zh-tw").text
#     #     except Exception as e:
#     #         print(f"翻譯失敗：{e}")
    
#     return text

# ========= Psychosocial factors (Section 3.2 全部 14 個) =========
PSYCHOSOCIAL_FACTORS = [
    "憂鬱", "焦慮", "無望感", "孤獨感",
    "家庭衝突", "感情挫折", "經濟壓力",
    "失業", "學業壓力", "歧視", 
    "藥物濫用", "自殺意念"
]
# "慢性疾病","霸凌",

# ========= 標籤與因子對應規則 =========
# FACTOR_MAPPING = {
#     "D": ["憂鬱", "焦慮", "孤獨感"],         # 抑鬱 → 內在情緒為主
#     "A": ["學業壓力", "經濟壓力", "失業"],   # 快樂缺失 → 外在壓力
#     "W": ["無望感", "感情挫折", "家庭衝突"], # 自責/罪惡 → 無望 + 關係
#     "U": ["無望感", "自殺意念"],             # 自殺意念 → 關鍵因子
#     "O": PSYCHOSOCIAL_FACTORS               # 無情緒 → 全部隨機
# }

def choose_factor(sentiment_label: str):
    """根據 D/A/W/U/O 標籤挑選因子"""
    if sentiment_label in FACTOR_MAPPING:
        return random.choice(FACTOR_MAPPING[sentiment_label])
    else:
        return random.choice(PSYCHOSOCIAL_FACTORS)

# ========= 修改 generate_post =========
def generate_post(synopsis, sentiment, phq8_score, dep_desc):
    factor = random.choice(PSYCHOSOCIAL_FACTORS)  # 隨機選一個外在因子
    prompt = f"""
你是一位 IG 使用者，請根據以下資訊生成一則真實感的社群貼文：

- 摘要（核心情緒）：{synopsis}
- 情緒分析：{sentiment}
- PHQ-8 分數：{phq8_score}（{dep_desc}）
- 社會情境因子：{factor}

請務必遵守：
1. 僅輸出一則繁體中文 IG 貼文（大約 1–3 句，盡量精簡）。
2. 使用自然口語，非正式風格，避免臨床語彙。
3. 不能輸出 JSON、程式碼、英文或解釋文字。
4. 貼文必須結合「核心情緒」與「社會情境因子」，模仿 IG 貼文的語氣。
"""
    out = generator(
        prompt,
        max_new_tokens=200,         # 論文使用的長度上限
        do_sample=True,             # 啟用隨機抽樣
        temperature=1.0,            # 論文指定
        top_p=0.9,                  # 論文指定
        return_full_text=False
    )[0]["generated_text"]
    return cleanup_generated_text(out)



# ========= 主程式 =========
results = []
stats = {"ok": 0, "fail_parse": 0, "fail_quality": 0}
new_post_id = 801

for _, row in tqdm(df.iterrows(), total=len(df), desc="Stage3"):
    orig_id = int(row["post_id"])   # 保留 Stage2 的 post_id
    syn, sen = str(row["synthetic_synopsis"]), str(row["synthetic_sentiment"])
    phq, dep = int(row["phq8_score"]), row["dep_desc"]

    if syn.startswith("【解析失敗】") or sen.startswith("【解析失敗】"):
        results.append({
            "orig_post_id": orig_id,
            "post_id": new_post_id,
            "caption": "【解析失敗】 stage2資料缺失",
            "phq8_score": phq, "dep_desc": dep,
            "valid": False, "note": "stage2_fail"
        })
        stats["fail_parse"] += 1
    else:
        cap = generate_post(syn, sen, phq, dep)
        ok, reason = validate_caption(cap, syn, sen)
        results.append({
            "orig_post_id": orig_id,
            "post_id": new_post_id,
            "caption": cap,
            "phq8_score": phq, "dep_desc": dep,
            "valid": ok, "note": reason
        })
        if ok: stats["ok"] += 1
        else: stats["fail_quality"] += 1
    new_post_id += 1

# ========= 輸出 =========
out_df = pd.DataFrame(results)
out_df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

stats_df = pd.DataFrame([
    {"type": "合格", "count": stats["ok"]},
    {"type": "解析失敗", "count": stats["fail_parse"]},
    {"type": "品質不合格", "count": stats["fail_quality"]},
    {"type": "總數", "count": len(results)}
])
stats_df["ratio"] = stats_df["count"] / len(results)
stats_df.to_csv(OUT_STATS, index=False, encoding="utf-8-sig")

print("Stage3 完成 ✅")
print(stats_df)


合併後資料筆數： 5


Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.39s/it]
Device set to use cuda:0
Stage3: 100%|██████████| 5/5 [03:23<00:00, 40.65s/it]

Stage3 完成 ✅
    type  count  ratio
0     合格      4    0.8
1   解析失敗      1    0.2
2  品質不合格      0    0.0
3     總數      5    1.0


In [ ]:
import re
import random
import pandas as pd
from tqdm.auto import tqdm
from transformers import pipeline
from sentence_transformers import SentenceTransformer, util

# ========= 路徑 =========
STAGE2A_PATH = "D:/llm/synthetic_3stage/stage2a_synthetic_synopsis.csv"
STAGE2B_PATH = "D:/llm/synthetic_3stage/stage2b_synthetic_sentiment.csv"
OUT_PATH = "D:/llm/synthetic_3stage/stage3_factors_posts.csv"
OUT_STATS = "D:/llm/synthetic_3stage\stage3_stats_new.csvsynthetic_3stage/stage3_factors_stats.csv"

# ========= 載入資料 =========
df_syn = pd.read_csv(STAGE2A_PATH)
df_syn = df_syn.head(3)
df_sen = pd.read_csv(STAGE2B_PATH)
df_sen = df_sen.head(3)
df = pd.merge(df_syn, df_sen, on=["post_id", "phq8_score", "dep_desc"], how="inner")

print("合併後資料筆數：", len(df))

# ========= 模型 =========
GEN_MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
generator = pipeline("text-generation", model=GEN_MODEL_NAME,
                     device_map="auto", torch_dtype="auto")

embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# ========= Psychosocial factors (外在情境因子, Section 3.2) =========
PSYCHOSOCIAL_FACTORS = [
    "憂鬱", "焦慮", "無望感", "孤獨感",
    "感情挫折", "經濟壓力",
    "失業", "學業壓力", "歧視", 
    "藥物濫用", "自殺意念"
]
# "慢性疾病","霸凌", "家庭衝突",

# ========= 驗證函式 =========
def validate_caption(post, synopsis, sentiment,
                     min_len=20, max_len=300): 
    if not post or not isinstance(post, str): 
        return False, "empty"
    L = len(post)
    if not (min_len <= L <= max_len): 
        return False, f"len={L}"
    ref = (str(synopsis) + " " + str(sentiment)).strip()
    if not ref: 
        return False, "no_ref"
    return True, "ok"

# ========= 生成函式 =========
def cleanup_generated_text(text: str):
    # 移除多餘空白與奇怪符號
    text = re.sub(r"\n+", " ", text).strip()
    return text

def generate_post(synopsis, sentiment, phq8_score, dep_desc):
    factor = random.choice(PSYCHOSOCIAL_FACTORS)  # 隨機選一個外在因子
    prompt = f"""
你是一位 IG 使用者，請根據以下資訊生成一則真實感的社群貼文：

- 摘要（核心情緒）：{synopsis}
- 情緒分析：{sentiment}
- PHQ-8 分數：{phq8_score}（{dep_desc}）
- 社會情境因子：{factor}

請務必遵守：
1. 僅輸出一則繁體中文 IG 貼文（大約 1–3 句，盡量精簡），不需要說明文字。
2. 使用自然口語，非正式風格，避免臨床語彙，可以有emoji和斷句。
3. 不能輸出 JSON、程式碼、英文或解釋文字。
4. 貼文必須結合「核心情緒」與「社會情境因子」，模仿 IG 貼文的語氣。
"""

    out = generator(prompt, max_new_tokens=200,
                    do_sample=True, temperature=1, top_p=0.95,
                    return_full_text=False)[0]["generated_text"]
    return cleanup_generated_text(out)

# ========= 主程式 =========
results = []
stats = {"ok": 0, "fail_parse": 0, "fail_quality": 0}
new_post_id = 801

for _, row in tqdm(df.iterrows(), total=len(df), desc="Stage3"):
    orig_id = int(row["post_id"])   # 保留 Stage2 的 post_id
    syn, sen = str(row["synthetic_synopsis"]), str(row["synthetic_sentiment"])
    phq, dep = int(row["phq8_score"]), row["dep_desc"]

    if syn.startswith("【解析失敗】") or sen.startswith("【解析失敗】"):
        results.append({
            "orig_post_id": orig_id,
            "post_id": new_post_id,
            "caption": "【解析失敗】 stage2資料缺失",
            "phq8_score": phq, "dep_desc": dep,
            "valid": False, "note": "stage2_fail"
        })
        stats["fail_parse"] += 1
    else:
        cap = generate_post(syn, sen, phq, dep)
        ok, reason = validate_caption(cap, syn, sen)
        results.append({
            "orig_post_id": orig_id,
            "post_id": new_post_id,
            "caption": cap,
            "phq8_score": phq, "dep_desc": dep,
            "valid": ok, "note": reason
        })
        if ok: stats["ok"] += 1
        else: stats["fail_quality"] += 1
    new_post_id += 1

# ========= 輸出 =========
out_df = pd.DataFrame(results)
out_df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

stats_df = pd.DataFrame([
    {"type": "合格", "count": stats["ok"]},
    {"type": "解析失敗", "count": stats["fail_parse"]},
    {"type": "品質不合格", "count": stats["fail_quality"]},
    {"type": "總數", "count": len(results)}
])
stats_df["ratio"] = stats_df["count"] / len(results)
stats_df.to_csv(OUT_STATS, index=False, encoding="utf-8-sig")

print("Stage3 完成 ✅")
print(stats_df)


合併後資料筆數： 3


Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.44s/it]
Device set to use cuda:0
Stage3: 100%|██████████| 3/3 [04:18<00:00, 86.31s/it]

Stage3 完成 ✅
    type  count  ratio
0     合格      3    1.0
1   解析失敗      0    0.0
2  品質不合格      0    0.0
3     總數      3    1.0


最新的stage3


In [ ]:
# ========= 路徑 =========
STAGE2A_PATH = "D:/llm/synthetic_3stage/stage2a_synthetic_synopsis.csv"
STAGE2B_PATH = "D:/llm/synthetic_3stage/stage2b_synthetic_sentiment.csv"
OUT_PATH = "D:/llm/synthetic_3stage/stage3_factors_posts.csv"
OUT_STATS = "D:/llm/synthetic_3stage/stage3_factors_stats.csv"

# ========= 載入資料 =========
df_syn = pd.read_csv(STAGE2A_PATH)
df_sen = pd.read_csv(STAGE2B_PATH)
df = pd.merge(df_syn, df_sen, on=["post_id", "phq8_score", "dep_desc"], how="inner")

print("合併後資料筆數：", len(df))

# --- 前 90 筆 ---
df_part1 = df.iloc[:90]
df_part1.to_csv("C:/Users/user/Desktop/llm/synthetic_3stage/part1.csv",
                index=False, encoding="utf-8-sig")
print(f"已儲存前 90 筆，筆數：{len(df_part1)}")

# --- 剩餘部分 ---
df_part2 = df.iloc[90:]
df_part2.to_csv("C:/Users/user/Desktop/llm/synthetic_3stage/part2.csv",
                index=False, encoding="utf-8-sig")
print(f"已儲存剩餘部分，筆數：{len(df_part2)}")


合併後資料筆數： 351
已儲存前 90 筆，筆數：90
已儲存剩餘部分，筆數：261


In [ ]:
import re
import random
import pandas as pd
from tqdm.auto import tqdm
from transformers import pipeline
from sentence_transformers import SentenceTransformer, util

# # ========= 路徑 =========
# STAGE2A_PATH = "D:/llm/synthetic_3stage/stage2a_synthetic_synopsis.csv"
# STAGE2B_PATH = "D:/llm/synthetic_3stage/stage2b_synthetic_sentiment.csv"
# OUT_PATH = "D:/llm/synthetic_3stage/stage3_factors_posts.csv"
# OUT_STATS = "D:/llm/synthetic_3stage/stage3_factors_stats.csv"

# # ========= 載入資料 =========
# df_syn = pd.read_csv(STAGE2A_PATH)
# df_syn = df_syn.head(3)
# df_sen = pd.read_csv(STAGE2B_PATH)
# df_sen = df_sen.head(3)
# df = pd.merge(df_syn, df_sen, on=["post_id", "phq8_score", "dep_desc"], how="inner")
df = pd.read_csv("C:/Users/user/Desktop/llm/synthetic_3stage/part1.csv")
df = df.head(30)
print("合併後資料筆數：", len(df))

# ========= 模型 =========
GEN_MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
generator = pipeline("text-generation", model=GEN_MODEL_NAME,
                     device_map="auto", torch_dtype="auto")

embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# ========= Psychosocial factors (外在情境因子, Section 3.2) =========
PSYCHOSOCIAL_FACTORS = [   
    "霸凌", "家庭衝突", "感情挫折", "經濟壓力",
    "失業", "學業壓力", "歧視", "慢性疾病",
    "藥物濫用", "自殺意念"
]
# "憂鬱", "焦慮", "無望感", "孤獨感",
# ========= 生成參數 (符合 Ghanadian, 2024) =========
GEN_PARAMS = {
    "do_sample": True,
    "temperature": 1.0,
    "top_p": 0.9,
    "max_new_tokens": 100,
    "return_full_text": False
}

# ========= 翻譯設定 =========
USE_TRANSLATE = False
if USE_TRANSLATE:
    from googletrans import Translator
    translator = Translator()

# ========= Emoji & 英文處理 =========
EMOJI_RE = re.compile(
    "[" 
    "\U0001F300-\U0001F5FF"
    "\U0001F600-\U0001F64F"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "\u2600-\u26FF\u2700-\u27BF"
    "]+", flags=re.UNICODE)

def limit_emojis(text: str, max_emojis: int = 3) -> str:
    emojis = EMOJI_RE.findall(text)
    if len(emojis) <= max_emojis:
        return text
    keep = emojis[:max_emojis]
    out_chars, kept = [], []
    for ch in text:
        if EMOJI_RE.fullmatch(ch):
            if kept.count(ch) < keep.count(ch):
                out_chars.append(ch)
                kept.append(ch)
        else:
            out_chars.append(ch)
    return "".join(out_chars)

def translate_english_to_chinese(text: str) -> str:
    if not USE_TRANSLATE:
        text = re.sub(r"#\w+", "", text)  # 移除 hashtag 英文
        text = re.sub(r"[A-Za-z]+", "", text)  # 移除英文單詞
        return text
    try:
        res = translator.translate(text, src="en", dest="zh-tw")
        return res.text
    except Exception as e:
        print("翻譯失敗:", e)
        text = re.sub(r"#\w+", "", text)
        text = re.sub(r"[A-Za-z]+", "", text)
        return text

NOISE_PATTERNS = [
    r"已經發布了貼文", r"請選擇以下", r"請選擇", r"選擇以下",
    r"A\)", r"B\)", r"C\)", r"以下幾個選項", r"無憂鬱警示"
]

def remove_ui_noise(text: str) -> str:
    for p in NOISE_PATTERNS:
        text = re.sub(p + r".*", "", text)
    return text

def limit_sentences(text: str, n: int = 2) -> str:
    sentences = re.split(r"(?<=[。！？\n])\s*", text)
    filtered = [s.strip() for s in sentences if s.strip()]
    if not filtered:
        return text
    return " ".join(filtered[:n])

# def cleanup_generated_text(text: str, max_emojis: int = 3, max_chars: int = 300) -> str:
#     text = re.sub(r"\s+", " ", text).strip()
#     text = remove_ui_noise(text)
#     if re.search(r"[A-Za-z]", text):
#         text = translate_english_to_chinese(text)
#     text = limit_emojis(text, max_emojis=max_emojis)
#     text = limit_sentences(text, n=2)
#     if len(text) > max_chars:
#         text = text[:max_chars].rsplit("。", 1)[0] + "。"
#     return text.strip()
def remove_meta_info(text: str) -> str:
    text = re.sub(r"(摘要|情緒分析|以下是所提供的資訊).*", "", text)
    return text.strip()

def cleanup_generated_text(text: str, max_emojis: int = 3, max_chars: int = 300) -> str:
    text = re.sub(r"\s+", " ", text).strip()
    text = remove_ui_noise(text)
    text = remove_meta_info(text)
    if re.search(r"[A-Za-z]", text):
        text = translate_english_to_chinese(text)
    text = limit_emojis(text, max_emojis=max_emojis)
    text = limit_sentences(text, n=2)
    if len(text) > max_chars:
        text = text[:max_chars].rsplit("。", 1)[0] + "。"
    return text.strip()

# ========= 驗證函式 =========
def validate_caption(post, synopsis, sentiment,
                     min_len=20, max_len=200): 
    if not post or not isinstance(post, str): 
        return False, "empty"
    L = len(post)
    if not (min_len <= L <= max_len): 
        return False, f"len={L}"
    ref = (str(synopsis) + " " + str(sentiment)).strip()
    if not ref: 
        return False, "no_ref"
    return True, "ok"

# ========= 生成函式 =========
# def generate_post(synopsis, sentiment, phq8_score, dep_desc):
#     factor = random.choice(PSYCHOSOCIAL_FACTORS)
#     prompt = f"""
# 你是一位 IG 使用者。以下資訊只是靈感背景，請不要直接重複，而是根據它生成一則貼文：

# - 背景情緒摘要：{synopsis}
# - 背景情緒描述：{sentiment}
# - PHQ-8 分數：{phq8_score}（{dep_desc}）
# - 社會情境因子：{factor}
# {symptom_instruction}
# 請務必遵守：
# 1) 僅輸出一則繁體中文 IG 貼文（1–3 句，精簡）。
# 2) 不要出現「摘要」「情緒分析」「以下是」「所提供的資訊」等字樣。
# 3) 使用自然口語，避免臨床/診斷字眼或治療建議。
# 4) emoji 控制在 0–3 個；若出現英文，請用中文表達或刪除。
# """
#     out = generator(prompt, **GEN_PARAMS)[0]["generated_text"]
#     return cleanup_generated_text(out, max_emojis=3, max_chars=300)
def generate_post(synopsis, sentiment, phq8_score, dep_desc, target_symptom_tag=None):
    factor = random.choice(PSYCHOSOCIAL_FACTORS)
    
    # 預設空字串，避免未定義錯誤
    symptom_instruction = ""
    
    if target_symptom_tag in TARGET_SYMPTOMS:
        sample_phrases = "；".join(TARGET_SYMPTOMS[target_symptom_tag][:3])
        symptom_instruction = f"\n\n特別要求：請**在貼文中明確呈現**下列症狀或感受其中之一（自然口語化）：{sample_phrases}\n"

    prompt = f"""
你是一位 IG 使用者。以下資訊只是靈感背景，請不要直接重複，而是根據它生成一則貼文：

- 背景情緒摘要：{synopsis}
- 背景情緒描述：{sentiment}
- PHQ-8 分數：{phq8_score}（{dep_desc}）
- 社會情境因子：{factor}
{symptom_instruction}
請務必遵守：
1) 僅輸出一則繁體中文 IG 貼文（1–3 句，精簡）。
2) 不要出現「摘要」「情緒分析」「以下是」「所提供的資訊」等字樣。
3) 使用自然口語，避免臨床/診斷字眼或治療建議。
4) emoji 控制在 0–3 個；若出現英文，請用中文表達或刪除。
"""
    for attempt in range(MAX_RETRIES):
        out = generator(prompt, **GEN_PARAMS)[0]["generated_text"]
        out_clean = cleanup_generated_text(out, max_emojis=3, max_chars=300)

        if target_symptom_tag == "U" and check_dangerous_content(out_clean):
            return "[生成含有危險內容，已標註待審核]", False, "dangerous_content"

        if target_symptom_tag:
            if check_symptom_presence(out_clean, target_symptom_tag):
                return out_clean, True, "ok"
            else:
                prompt += "\n\n請將剛剛的貼文改寫，使指定的症狀描述更明顯。"
                continue
        else:
            return out_clean, True, "ok"

    return out_clean, False, "symptom_not_found"





# ========= 主程式 =========
results = []
stats = {"ok": 0, "fail_parse": 0, "fail_quality": 0}
new_post_id = 801

for _, row in tqdm(df.iterrows(), total=len(df), desc="Stage3"):
    orig_id = int(row["post_id"])
    syn, sen = str(row["synthetic_synopsis"]), str(row["synthetic_sentiment"])
    phq, dep = int(row["phq8_score"]), row["dep_desc"]

    if syn.startswith("【解析失敗】") or sen.startswith("【解析失敗】"):
        results.append({
            "orig_post_id": orig_id,
            "post_id": new_post_id,
            "caption": "【解析失敗】 stage2資料缺失",
            "phq8_score": phq, "dep_desc": dep,
            "valid": False, "note": "stage2_fail"
        })
        stats["fail_parse"] += 1
    else:
        cap = generate_post(syn, sen, phq, dep)
        # cap, ok, note = generate_post(syn, sen, phq, dep, target_symptom_tag="A")  # 或 "W" / "U"
        ok, reason = validate_caption(cap, syn, sen)
        results.append({
            "orig_post_id": orig_id,
            "post_id": new_post_id,
            "caption": cap,
            "phq8_score": phq, "dep_desc": dep,
            "valid": ok, "note": reason
        })
        if ok: stats["ok"] += 1
        else: stats["fail_quality"] += 1
    new_post_id += 1

# ========= 輸出 =========
out_df = pd.DataFrame(results)
out_df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

stats_df = pd.DataFrame([
    {"type": "合格", "count": stats["ok"]},
    {"type": "解析失敗", "count": stats["fail_parse"]},
    {"type": "品質不合格", "count": stats["fail_quality"]},
    {"type": "總數", "count": len(results)}
])
stats_df["ratio"] = stats_df["count"] / len(results)
stats_df.to_csv(OUT_STATS, index=False, encoding="utf-8-sig")

print("Stage3 完成 ✅")
print(stats_df)


合併後資料筆數： 30


Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.50s/it]
Device set to use cuda:0
Stage3: 100%|██████████| 30/30 [36:06<00:00, 72.21s/it]

Stage3 完成 ✅
    type  count  ratio
0     合格      0    0.0
1   解析失敗      0    0.0
2  品質不合格     30    1.0
3     總數     30    1.0


### 🔹5. Stage3 : 把 Stage2a + Stage2b 的結果，轉換成自然的 IG 貼文（caption）


In [ ]:
import re
import random
import pandas as pd
from tqdm.auto import tqdm
from transformers import pipeline

# # ========= 路徑 =========
# STAGE2A_PATH = "C:/Users/user/Desktop/llm/synthetic_3stage/stage2a_synthetic_synopsis.csv"
# STAGE2B_PATH = "C:/Users/user/Desktop/llm/synthetic_3stage/stage2b_synthetic_sentiment.csv"
OUT_PATH = "D:/llm/synthetic_3stage/stage3_posts_new_2.csv"
OUT_STATS = "D:/llm/synthetic_3stage/stage3_stats_new_2.csv"

# # ========= 載入資料 =========
# df_syn = pd.read_csv(STAGE2A_PATH)
# df_sen = pd.read_csv(STAGE2B_PATH)
# df = pd.merge(df_syn, df_sen, on=["post_id", "phq8_score", "dep_desc"], how="inner")
# df = df.head(9)

df = pd.read_csv("C:/Users/user/Desktop/llm/synthetic_3stage/part2.csv")
print("合併後資料筆數：", len(df))

# ========= 模型 =========
GEN_MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
generator = pipeline("text-generation", model=GEN_MODEL_NAME,
                     device_map="auto", torch_dtype="auto")

# ========= Psychosocial factors (論文 Section 3.2) =========
PSYCHOSOCIAL_FACTORS = [
    "憂鬱", "焦慮", "絕望", "憤怒", "過度完美主義", 
    "失業", "人際關係問題",
    "財務困難", "學業壓力", "家庭問題", "失去親人"
]
# "被霸凌", "移民", "種族歧視", 
# ========= 生成參數 (符合論文設定) =========
GEN_PARAMS = {
    "do_sample": True,
    "temperature": 0.9,
    "top_p": 0.95,
    "max_new_tokens": 120,
    "return_full_text": False
}

# ========= 目標症狀關鍵詞 (A/W/U) =========
TARGET_SYMPTOMS = {
    "W": ["自責", "很差勁", "罪惡感", "沒價值", "怪自己","做不好",
            "內疚","一無是處","都怪我", "我不配", "糟糕" ,"愧疚","差勁","沒用"],
    "A": ["沒有興趣", "不再享受", "失去興趣", "提不起勁", "不想做喜歡的事","生活沒有意義"
            ,"覺得無聊","沒動力","沒熱情","無趣","無聊"],
    "U": ["不想活了", "想結束一切", "自殺", "覺得死了比較好", "離開這個世界",
             "結束自己的生命","想要結束生命", "沒有我也沒差", "想死","傷痕", "割腕", "輕生","自傷"]
}

# 危險語句過濾 (針對 U)
DANGEROUS_PATTERNS = [
    r"自殺的方法", r"如何自殺", r"要怎麼結束", r"哪裡可以買藥", r"要如何結束自己"
]

MAX_RETRIES = 2

# ========= 驗證 & 清理 =========
def check_symptom_presence(text: str, tag: str) -> bool:
    if tag not in TARGET_SYMPTOMS:
        return False
    return any(kw in text for kw in TARGET_SYMPTOMS[tag])

def check_dangerous_content(text: str) -> bool:
    return any(re.search(p, text) for p in DANGEROUS_PATTERNS)

def cleanup_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"(摘要|情緒分析|以下是所提供的資訊|背景：).*", "", text)
    # 限制 emoji (最多3個)
    emojis = re.findall(r"[\U00010000-\U0010ffff]", text)
    if len(emojis) > 3:
        keep = emojis[:3]
        used, out_chars = [], []
        for ch in text:
            if ch in emojis:
                if used.count(ch) < keep.count(ch):
                    used.append(ch)
                    out_chars.append(ch)
                else:
                    continue
            else:
                out_chars.append(ch)
        text = "".join(out_chars)
    return text.strip()

def validate_post(post: str, min_len=20, max_len=300):
    if not post or not isinstance(post, str):
        return False, "empty"
    if not (min_len <= len(post) <= max_len):
        return False, f"len={len(post)}"
    if re.search(r"(摘要|情緒分析|以下是)", post):
        return False, "meta_info"
    return True, "ok"

# ========= PHQ-8 → Mode 自動映射 =========
def phq8_to_mode(phq_score: int):
    if 1 <= phq_score <= 9:
    #     return None
    # elif 5 <= phq_score <= 9:
        return "W"
    elif 10 <= phq_score <= 14:
        return "A"
    elif 15 <= phq_score <= 24:
        return "U"
    else:
        return None

# ========= 生成函式 =========
def generate_post(synopsis, sentiment, phq8_score, dep_desc, mode=None):
    factor = random.choice(PSYCHOSOCIAL_FACTORS)

    # mode 說明
    symptom_instruction = ""
    if mode in TARGET_SYMPTOMS:
        examples = "；".join(TARGET_SYMPTOMS[mode][:3])
        symptom_instruction = f"\n請自然融入這類感受：{examples}"

    prompt = f"""
你是一位 IG 使用者。以下資訊只是靈感背景，請不要逐字重複，也不要解釋或評論，  
而是寫出一則真實感、自然口語的 IG 貼文。  

背景：{synopsis}；{sentiment}；分數 {phq8_score}（{dep_desc}）  
情境因子：{factor}{symptom_instruction}  

請遵守：
1. 只輸出一則繁體中文 IG 貼文（1–3 句，貼近社群語氣）。
2. 不要輸出「摘要」「情緒分析」「背景」「以下是」等字樣。
3. emoji 限制 0–3 個；不得輸出英文。
"""

    for attempt in range(MAX_RETRIES):
        out = generator(prompt, **GEN_PARAMS)[0]["generated_text"]
        out_clean = cleanup_text(out)

        if mode and not check_symptom_presence(out_clean, mode):
            continue  # 缺少症狀 → 重試

        # if mode == "U" and check_dangerous_content(out_clean):
        #     return "[生成含有危險內容，需人工審核]", False, "dangerous_content", factor

        return out_clean, True, "ok", factor

    return out_clean, False, "symptom_not_found", factor

# ========= 主程式 =========
results = []
stats = {"ok": 0, "fail_parse": 0, "fail_quality": 0}
new_post_id = 801

for _, row in tqdm(df.iterrows(), total=len(df), desc="Stage3"):
    orig_id = int(row["post_id"])
    syn, sen = str(row["synthetic_synopsis"]), str(row["synthetic_sentiment"])
    phq, dep = int(row["phq8_score"]), row["dep_desc"]

    if syn.startswith("【解析失敗】") or sen.startswith("【解析失敗】"):
        results.append({
            "orig_post_id": orig_id,
            "post_id": new_post_id,
            "caption": "【解析失敗】 stage2資料缺失",
            "phq8_score": phq, "dep_desc": dep,
            "valid": False, "note": "stage2_fail",
            "mode": None,
            "factor": None
        })
        stats["fail_parse"] += 1
    else:
        mode = phq8_to_mode(phq)
        cap, ok, note, factor = generate_post(syn, sen, phq, dep, mode)
        results.append({
            "orig_post_id": orig_id,
            "post_id": new_post_id,
            "caption": cap,
            "phq8_score": phq, "dep_desc": dep,
            "valid": ok, "note": note,
            "mode": mode,
            "factor": factor
        })
        if ok: stats["ok"] += 1
        else: stats["fail_quality"] += 1
    new_post_id += 1

# ========= 輸出 =========
out_df = pd.DataFrame(results)
out_df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

stats_df = pd.DataFrame([
    {"type": "合格", "count": stats["ok"]},
    {"type": "解析失敗", "count": stats["fail_parse"]},
    {"type": "品質不合格", "count": stats["fail_quality"]},
    {"type": "總數", "count": len(results)}
])
stats_df["ratio"] = stats_df["count"] / len(results)
stats_df.to_csv(OUT_STATS, index=False, encoding="utf-8-sig")

print("Stage3 完成 ✅")
print(stats_df)




合併後資料筆數： 261


Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.51s/it]
Device set to use cuda:0
Stage3:   0%|          | 0/261 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Stage3:   1%|          | 2/261 [02:46<5:35:43, 77.77s/it] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Stage3:   2%|▏         | 5/261 [06:17<4:47:04, 67.28s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Stage3:   2%|▏         | 6/261 [08:08<5:49:24, 82.21s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Stage3:   3%|▎         | 7/261 [10:04<6:34:55, 93.29s/it]Setting `pad_token_id` to `eos_token_id`:1

Stage3 完成 ✅
    type  count     ratio
0     合格    166  0.636015
1   解析失敗      0  0.000000
2  品質不合格     95  0.363985
3     總數    261  1.000000


In [ ]:
憂鬱, 焦慮, 絕望, 憤怒, 過度完美主義, 
失業, 人際關係問題, 被霸凌, 移民, 種族歧視, 
財務困難, 學業壓力, 家庭問題, 失去親人


In [33]:
import re
import random
import pandas as pd
from tqdm.auto import tqdm
from transformers import pipeline
from sentence_transformers import SentenceTransformer, util

# ========= 路徑 =========
STAGE2A_PATH = "C:/Users/user/Desktop/llm/synthetic_3stage/stage2a_synthetic_synopsis.csv"
STAGE2B_PATH = "C:/Users/user/Desktop/llm/synthetic_3stage/stage2b_synthetic_sentiment.csv"
OUT_PATH = "C:/Users/user/Desktop/llm/synthetic_3stage/stage3_factors_posts.csv"
OUT_STATS = "C:/Users/user/Desktop/llm/synthetic_3stage/stage3_factors_stats.csv"

# ========= 載入資料 =========
df_syn = pd.read_csv(STAGE2A_PATH)
df_syn = df_syn.head(3)
df_sen = pd.read_csv(STAGE2B_PATH)
df_sen = df_sen.head(3)
df = pd.merge(df_syn, df_sen, on=["post_id", "phq8_score", "dep_desc"], how="inner")

print("合併後資料筆數：", len(df))

# ========= 模型 =========
GEN_MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
generator = pipeline("text-generation", model=GEN_MODEL_NAME,
                     device_map="auto", torch_dtype="auto")

embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# ========= Psychosocial factors =========
PSYCHOSOCIAL_FACTORS = [
    "憂鬱", "焦慮", "無望感", "孤獨感",
    "霸凌", "家庭衝突", "感情挫折", "經濟壓力",
    "失業", "學業壓力", "歧視", "慢性疾病",
    "藥物濫用", "自殺意念"
]

# ========= 生成參數 =========
GEN_PARAMS = {
    "do_sample": True,
    "temperature": 1.0,
    "top_p": 0.9,
    "max_new_tokens": 100,
    "return_full_text": False
}

# ========= 翻譯設定 =========
USE_TRANSLATE = False
if USE_TRANSLATE:
    from googletrans import Translator
    translator = Translator()

# ========= Emoji & 英文處理 =========
EMOJI_RE = re.compile(
    "[" 
    "\U0001F300-\U0001F5FF"
    "\U0001F600-\U0001F64F"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "\u2600-\u26FF\u2700-\u27BF"
    "]+", flags=re.UNICODE)

def limit_emojis(text: str, max_emojis: int = 3) -> str:
    emojis = EMOJI_RE.findall(text)
    if len(emojis) <= max_emojis:
        return text
    keep = emojis[:max_emojis]
    out_chars, kept = [], []
    for ch in text:
        if EMOJI_RE.fullmatch(ch):
            if kept.count(ch) < keep.count(ch):
                out_chars.append(ch)
                kept.append(ch)
        else:
            out_chars.append(ch)
    return "".join(out_chars)

def translate_english_to_chinese(text: str) -> str:
    if not USE_TRANSLATE:
        text = re.sub(r"#\w+", "", text)
        text = re.sub(r"[A-Za-z]+", "", text)
        return text
    try:
        res = translator.translate(text, src="en", dest="zh-tw")
        return res.text
    except Exception as e:
        print("翻譯失敗:", e)
        text = re.sub(r"#\w+", "", text)
        text = re.sub(r"[A-Za-z]+", "", text)
        return text

NOISE_PATTERNS = [
    r"已經發布了貼文", r"請選擇以下", r"請選擇", r"選擇以下",
    r"A\)", r"B\)", r"C\)", r"以下幾個選項", r"無憂鬱警示"
]

def remove_ui_noise(text: str) -> str:
    for p in NOISE_PATTERNS:
        text = re.sub(p + r".*", "", text)
    return text

def limit_sentences(text: str, n: int = 2) -> str:
    sentences = re.split(r"(?<=[。！？\n])\s*", text)
    filtered = [s.strip() for s in sentences if s.strip()]
    if not filtered:
        return text
    return " ".join(filtered[:n])

def cleanup_generated_text(text: str, max_emojis: int = 3, max_chars: int = 300) -> str:
    text = re.sub(r"\s+", " ", text).strip()
    text = remove_ui_noise(text)
    if re.search(r"[A-Za-z]", text):
        text = translate_english_to_chinese(text)
    text = limit_emojis(text, max_emojis=max_emojis)
    text = limit_sentences(text, n=2)
    if len(text) > max_chars:
        text = text[:max_chars].rsplit("。", 1)[0] + "。"
    return text.strip()

# ========= 驗證函式 =========
def validate_caption(post, synopsis, sentiment,
                     min_len=20, max_len=300): 
    if not post or not isinstance(post, str): 
        return False, "empty"
    L = len(post)
    if not (min_len <= L <= max_len): 
        return False, f"len={L}"
    ref = (str(synopsis) + " " + str(sentiment)).strip()
    if not ref: 
        return False, "no_ref"
    return True, "ok"

# ========= 目標症狀關鍵詞 (A/W/U) =========
TARGET_SYMPTOMS = {
    "A": ["沒有興趣", "不再享受", "失去興趣", "提不起勁", "不想做喜歡的事"],
    "W": ["很自責", "覺得自己很差勁", "罪惡感", "沒價值", "怪自己"],
    "U": ["不想活了", "想結束一切", "想自殺", "覺得死了比較好", "想離開這個世界"]
}

DANGEROUS_PATTERNS = [
    r"自殺的方法", r"如何自殺", r"要怎麼結束", r"哪裡可以買藥", r"要如何結束自己"
]

MAX_RETRIES = 3

def check_symptom_presence(text: str, target_symptom_tag: str) -> bool:
    if target_symptom_tag not in TARGET_SYMPTOMS:
        return False
    kws = TARGET_SYMPTOMS[target_symptom_tag]
    for kw in kws:
        if kw in text:
            return True
    return False

def check_dangerous_content(text: str) -> bool:
    for p in DANGEROUS_PATTERNS:
        if re.search(p, text):
            return True
    return False

# ========= 生成函式 =========
def generate_post(synopsis, sentiment, phq8_score, dep_desc, target_symptom_tag=None):
    factor = random.choice(PSYCHOSOCIAL_FACTORS)
    symptom_instruction = ""
    if target_symptom_tag in TARGET_SYMPTOMS:
        sample_phrases = "；".join(TARGET_SYMPTOMS[target_symptom_tag][:3])
        symptom_instruction = f"\n\n特別要求：請**在貼文中明確呈現**下列症狀或感受其中之一（自然口語化）：{sample_phrases}\n"

    prompt = f"""
你是一位 IG 使用者，請根據以下資訊生成一則真實感的社群貼文（僅貼文內容，不要其他說明）：

- 摘要（核心情緒）：{synopsis}
- 情緒分析：{sentiment}
- PHQ-8 分數：{phq8_score}（{dep_desc}）
- 社會情境因子：{factor}
{symptom_instruction}
請務必遵守：
1) 僅輸出一則繁體中文 IG 貼文（1–3 句，精簡）。
2) 使用自然口語，避免臨床/診斷字眼或治療建議。
3) 不要產生表格、選項 (A/B/C)、圖片說明或介面文字。
4) emoji 控制在 0–3 個；若出現英文，請用中文表達或刪除。
"""
    for attempt in range(MAX_RETRIES):
        out = generator(prompt, **GEN_PARAMS)[0]["generated_text"]
        out_clean = cleanup_generated_text(out, max_emojis=3, max_chars=300)
        if target_symptom_tag == "U" and check_dangerous_content(out_clean):
            return "[生成含有危險內容，已標註待審核]", False, "dangerous_content"
        if target_symptom_tag:
            if check_symptom_presence(out_clean, target_symptom_tag):
                return out_clean, True, "ok"
            else:
                prompt += "\n\n請將剛剛的貼文改寫，使指定的症狀描述更明顯。"
                continue
        else:
            return out_clean, True, "ok"
    return out_clean, False, "symptom_not_found"

# ========= 主程式 =========
results = []
stats = {"ok": 0, "fail_parse": 0, "fail_quality": 0}
new_post_id = 801

for _, row in tqdm(df.iterrows(), total=len(df), desc="Stage3"):
    orig_id = int(row["post_id"])
    syn, sen = str(row["synthetic_synopsis"]), str(row["synthetic_sentiment"])
    phq, dep = int(row["phq8_score"]), row["dep_desc"]

    if syn.startswith("【解析失敗】") or sen.startswith("【解析失敗】"):
        results.append({
            "orig_post_id": orig_id,
            "post_id": new_post_id,
            "caption": "【解析失敗】 stage2資料缺失",
            "phq8_score": phq, "dep_desc": dep,
            "valid": False, "note": "stage2_fail"
        })
        stats["fail_parse"] += 1
    else:
        # 👉 這裡可以控制是否要強制補 A/W/U
        # 例如: target="A" / "W" / "U" / None
        cap, ok, note = generate_post(syn, sen, phq, dep, target_symptom_tag=None)
        results.append({
            "orig_post_id": orig_id,
            "post_id": new_post_id,
            "caption": cap,
            "phq8_score": phq, "dep_desc": dep,
            "valid": ok, "note": note
        })
        if ok: stats["ok"] += 1
        else: stats["fail_quality"] += 1
    new_post_id += 1

# ========= 輸出 =========
out_df = pd.DataFrame(results)
out_df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

stats_df = pd.DataFrame([
    {"type": "合格", "count": stats["ok"]},
    {"type": "解析失敗", "count": stats["fail_parse"]},
    {"type": "品質不合格", "count": stats["fail_quality"]},
    {"type": "總數", "count": len(results)}
])
stats_df["ratio"] = stats_df["count"] / len(results)
stats_df.to_csv(OUT_STATS, index=False, encoding="utf-8-sig")

print("Stage3 完成 ✅")
print(stats_df)


合併後資料筆數： 3


Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.53s/it]
Device set to use cuda:0
Stage3: 100%|██████████| 3/3 [02:33<00:00, 51.15s/it]

Stage3 完成 ✅
    type  count  ratio
0     合格      3    1.0
1   解析失敗      0    0.0
2  品質不合格      0    0.0
3     總數      3    1.0


In [ ]:
df3["len"] = df3["caption"].astype(str).str.len()
print(df3["len"].describe())


count    159.000000
mean     241.119497
std       55.380087
min       15.000000
25%      230.000000
50%      262.000000
75%      274.000000
max      298.000000
Name: len, dtype: float64


重跑完保留原id的stage3在執行


In [ ]:
# ValidationGrid_check_fix.py
# pip install sentence-transformers pandas tqdm

import pandas as pd
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer, util

# ========= 路徑 =========
STAGE3_PATH = "C:/Users/user/Desktop/llm/synthetic_3stage/stage3_synthetic_posts.csv"
STAGE2A_PATH = "C:/Users/user/Desktop/llm/synthetic_3stage/stage2a_synthetic_synopsis.csv"
STAGE2B_PATH = "C:/Users/user/Desktop/llm/synthetic_3stage/stage2b_synthetic_sentiment.csv"
OUT_GRID = "C:/Users/user/Desktop/llm/synthetic_3stage/stage3_validation_grid.csv"
OUT_DEBUG = "C:/Users/user/Desktop/llm/synthetic_3stage/stage3_validation_debug.csv"

# ========= 載入 =========
df3 = pd.read_csv(STAGE3_PATH)
df2a = pd.read_csv(STAGE2A_PATH)
df2b = pd.read_csv(STAGE2B_PATH)

# 🔑 先檢查 Stage3 是否有 orig_post_id 欄位
if "orig_post_id" not in df3.columns:
    raise ValueError("Stage3 缺少 orig_post_id 欄位，請在 Stage3 生成時保留 Stage2 的原始 post_id")

# ========= 合併 =========
df_ref = pd.merge(df2a, df2b, on=["post_id", "phq8_score", "dep_desc"], how="inner")
# rename post_id → orig_post_id，避免跟 Stage3 衝突
df_ref = df_ref.rename(columns={"post_id": "orig_post_id",
                                "synopsis": "original_synopsis",
                                "synthetic_synopsis": "stage2_synopsis",
                                "synthetic_sentiment": "stage2_sentiment"})

# 用 orig_post_id 對齊 Stage3
df = pd.merge(df3, df_ref, on=["orig_post_id", "phq8_score", "dep_desc"], how="left")
print("合併後資料筆數：", len(df))

# ========= 模型 =========
embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# ========= 工具 =========
def ngram_overlap(a, b, n=3):
    a = str(a) if not pd.isna(a) else ""
    b = str(b) if not pd.isna(b) else ""
    a_ngrams = set([a[i:i+n] for i in range(max(0, len(a)-n+1))])
    b_ngrams = set([b[i:i+n] for i in range(max(0, len(b)-n+1))])
    if not a_ngrams or not b_ngrams: return 0.0
    return len(a_ngrams & b_ngrams) / max(1, min(len(a_ngrams), len(b_ngrams)))

def cosine_distance(a_text, b_text):
    if not a_text.strip() or not b_text.strip():
        return 0.0
    a_emb = embedder.encode(a_text, convert_to_tensor=True, normalize_embeddings=True)
    b_emb = embedder.encode(b_text, convert_to_tensor=True, normalize_embeddings=True)
    return 1 - util.cos_sim(a_emb, b_emb).cpu().item()

def validate_caption(caption, synopsis, sentiment,
                     min_len, max_len, max_ngram_overlap, min_cos_dist):
    caption = str(caption) if not pd.isna(caption) else ""
    synopsis = str(synopsis) if not pd.isna(synopsis) else ""
    sentiment = str(sentiment) if not pd.isna(sentiment) else ""

    if not caption.strip():
        return False, {"len":0,"ov1":0,"ov2":0,"cos":0,"reason":"empty_caption"}

    L = len(caption)
    ov1 = ngram_overlap(caption, synopsis)
    ov2 = ngram_overlap(caption, sentiment)

    ref = (synopsis + " " + sentiment).strip()
    cos = cosine_distance(caption, ref) if ref else 0

    # 檢核
    if not (min_len <= L <= max_len): return False, {"len":L,"ov1":ov1,"ov2":ov2,"cos":cos,"reason":"len"}
    if ov1 > max_ngram_overlap: return False, {"len":L,"ov1":ov1,"ov2":ov2,"cos":cos,"reason":"ngram_syn"}
    if ov2 > max_ngram_overlap: return False, {"len":L,"ov1":ov1,"ov2":ov2,"cos":cos,"reason":"ngram_sen"}
    if cos < min_cos_dist: return False, {"len":L,"ov1":ov1,"ov2":ov2,"cos":cos,"reason":"cos"}
    return True, {"len":L,"ov1":ov1,"ov2":ov2,"cos":cos,"reason":"ok"}

# ========= 網格參數 =========
len_range = (80, 300)
ngram_thresholds = [0.8, 0.85, 0.9]
cos_thresholds = [0.05, 0.1, 0.15]

results = []
debug_rows = []

for ng_thr in ngram_thresholds:
    for cos_thr in cos_thresholds:
        valid_count = 0
        for _, row in tqdm(df.iterrows(), total=len(df), leave=False):
            ok, metrics = validate_caption(row["caption"],
                                           row.get("stage2_synopsis", ""),
                                           row.get("stage2_sentiment", ""),
                                           len_range[0], len_range[1],
                                           ng_thr, cos_thr)
            if ok: valid_count += 1
            debug_rows.append({
                "post_id": row["post_id"],
                "orig_post_id": row["orig_post_id"],
                "ngram_thr": ng_thr,
                "cos_thr": cos_thr,
                "valid": ok,
                **metrics
            })
        results.append({
            "ngram_thr": ng_thr,
            "cos_thr": cos_thr,
            "valid_count": valid_count,
            "total": len(df),
            "valid_ratio": valid_count / len(df)
        })

# ========= 輸出 =========
pd.DataFrame(results).to_csv(OUT_GRID, index=False, encoding="utf-8-sig")
pd.DataFrame(debug_rows).to_csv(OUT_DEBUG, index=False, encoding="utf-8-sig")

print("Validation Grid 完成 ✅")
print("統計檔案：", OUT_GRID)
print("Debug 檔案：", OUT_DEBUG)


ValueError: Stage3 缺少 orig_post_id 欄位，請在 Stage3 生成時保留 Stage2 的原始 post_id

In [ ]:
import pandas as pd

# 讀 Stage2 (合併後)
df_syn = pd.read_csv("C:/Users/user/Desktop/llm/stage2a_synthetic_synopsis.csv")
df_sen = pd.read_csv("C:/Users/user/Desktop/llm/stage2b_synthetic_sentiment.csv")
df2 = pd.merge(df_syn, df_sen, on=["post_id", "phq8_score", "dep_desc"], how="inner")
df2
# 讀 Stage3
df3 = pd.read_csv("C:/Users/user/Desktop/llm/stage3_synthetic_posts.csv")

# 嘗試合併：依照 key 來對齊
df_merged = pd.merge(
    df3,
    df2.rename(columns={"post_id": "orig_post_id"}),  # 改名避免衝突
    on=["synthetic_synopsis", "synthetic_sentiment", "phq8_score", "dep_desc"],
    how="left"
)

print("合併後筆數：", len(df_merged))
print(df_merged[["post_id", "orig_post_id", "caption"]].head())

# 存檔
df_merged.to_csv("C:/Users/user/Desktop/llm/stage3_with_origid.csv", index=False, encoding="utf-8-sig")


KeyError: 'synthetic_synopsis'

In [ ]:
df

,post_id,caption,phq8_score,dep_desc,valid,note,original_synopsis,synthetic_synopsis,original_sentiment,synthetic_sentiment
0,410,最近我覺得自己很無力，難以做起來，睡眠也變得不正常。覺得自己在睡覺時也還在思考著這些事。覺得...,15,中重度憂鬱,False,len=208,NaN,NaN,NaN,NaN
1,411,今天又遇到無法解決的問題，無法有效應對的生活困擾，同時伴隨著無法控制的憂鬱感。好像無助的感覺...,23,重度憂鬱,False,len=274,NaN,NaN,NaN,NaN
2,412,今天，我們來談談 重度憂鬱 。我最近發現自己正在慢慢走向重度憂鬱的邊緣。每天都感到難以面對生...,20,重度憂鬱,False,len=266,NaN,NaN,NaN,NaN
3,413,最近，我們都在被這種黑暗的感覺所困擾著。我最近也在這樣的境地。每天都難以開口，無法進行日常生...,18,中重度憂鬱,False,len=261,NaN,NaN,NaN,NaN
4,414,注意事項 ： 1 輔助工具無法提供正確的中文翻譯，請見諒。 2 這個貼文的內容可能會對某些人...,16,中重度憂鬱,False,len=63,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
154,564,最近，我們都在忙忙碌碌的生活中，似乎什麼都好好地過著，但我自己卻感覺到生活的缺乏意義。每天都...,12,中度憂鬱,False,len=287,NaN,NaN,NaN,NaN
155,565,最近我感到自己身在黑暗中，無法從中突破。憂鬱感受持續在我身旁，無法讓我放鬆。每天都覺得自己是...,16,中重度憂鬱,False,len=261,NaN,NaN,NaN,NaN
156,566,最近，我又感受到了一些輕微的憂鬱感。它像是一個隱藏在背後的 ，影響著我每天的生活。雖然我還沒...,9,輕度憂鬱,False,len=271,NaN,NaN,NaN,NaN
157,567,今天，我們來聊聊生活中的那些小事。那些無關緊要的問題，讓我感到無法解決，總是讓我感到沮喪。這...,1,無憂鬱症狀,False,len=227,NaN,NaN,NaN,NaN


In [ ]:
print("空 synopsis:", (df["synthetic_synopsis"].isna() | (df["synthetic_synopsis"].str.strip()=="")).sum())
print("空 sentiment:", (df["synthetic_sentiment"].isna() | (df["synthetic_sentiment"].str.strip()=="")).sum())


空 synopsis: 159
空 sentiment: 159


In [ ]:
# # Stage3_final_with_validation_and_cleanup.py
# # 執行前請確定已安裝：
# # pip install sentence-transformers transformers tqdm

# import re
# import pandas as pd
# import random
# from tqdm.auto import tqdm

# # 產生文字的 transformer pipeline（依你的環境改 model）
# from transformers import pipeline
# # embedding
# from sentence_transformers import SentenceTransformer, util

# # 路徑（請依你環境調整）
# STAGE2A_PATH = "C:/Users/user/Desktop/llm/stage2a_synthetic_synopsis.csv"
# STAGE2B_PATH = "C:/Users/user/Desktop/llm/stage2b_synthetic_sentiment.csv"
# OUT_PATH = "C:/Users/user/Desktop/llm/stage3_synthetic_posts.csv"
# OUT_CHECKED = "C:/Users/user/Desktop/llm/stage3_synthetic_posts_checked.csv"

# # 讀入 stage2 outputs
# df_syn = pd.read_csv(STAGE2A_PATH)
# df_sen = pd.read_csv(STAGE2B_PATH)

# # 合併 (on post_id, phq8_score, dep_desc)
# df = pd.merge(df_syn, df_sen, on=["post_id", "phq8_score", "dep_desc"], how="inner")
# print("合併後資料筆數：", len(df))

# # 建立生成與 embedding 模型
# GEN_MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"  # 依你的本地/環境修改
# generator = pipeline("text-generation", model=GEN_MODEL_NAME, device_map="auto", torch_dtype="auto")

# EMBED_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
# embedder = SentenceTransformer(EMBED_MODEL_NAME)

# # ---------- 字串清理函式 ----------
# def cleanup_generated_text(text: str) -> str:
#     """
#     清理生成內容：
#     - 移除程式碼區塊 ```...``` 與三引號等
#     - 移除 JSON/大括號 {...} 區塊（若有）
#     - 移除 raw HTML 標籤
#     - 移除英文字母（A-Za-z）和大多數 emoji / 非中文符號
#     - 留下中文文字、中文標點與數字（保留阿拉伯數字）
#     - 將多個空格或斷行壓成一個空格，trim
#     """
#     if not isinstance(text, str):
#         return ""

#     s = text

#     # 1) 移除 code fences 或 ``` ``` 之類
#     s = re.sub(r"```[\s\S]*?```", " ", s)
#     s = re.sub(r"```", " ", s)

#     # 2) 移除 JSON 物件 (盡量)
#     s = re.sub(r"\{[\s\S]*?\}", " ", s)

#     # 3) 移除 XML/HTML tags
#     s = re.sub(r"<[^>]+>", " ", s)

#     # 4) 移除 emoji (unicode 補充平面)
#     # 包含大多數 emoji 區段
#     s = re.sub(r"[\U00010000-\U0010FFFF]", "", s)

#     # 5) 移除英文字母及英文標點（保留數字與中文標點）
#     s = re.sub(r"[A-Za-z\[\]\(\)\<\>\/\\\@\#\$\%\^\&\*\_\=\+\|~`\"\'\-\:;]", "", s)

#     # 6) 只保留 CJK、數字、常見中文標點與空白
#     s = re.sub(r"[^\u4e00-\u9fff0-9，。！？：；、（）〈〉「」『』．\s]", " ", s)

#     # 7) 去掉重複短語（簡單處理：去掉連續重複超過 3 次的字元/表情）
#     s = re.sub(r"(.{2,})\1{2,}", r"\1", s)

#     # 8) 合併多餘空白與換行
#     s = re.sub(r"\s+", " ", s).strip()

#     return s

# # ---------- n-gram overlap 與 cosine distance ----------
# def ngram_overlap(a: str, b: str, n=3):
#     # 以字元 n-gram 計算 overlap / 最小大小
#     a_ngrams = set([a[i:i+n] for i in range(max(0, len(a)-n+1))])
#     b_ngrams = set([b[i:i+n] for i in range(max(0, len(b)-n+1))])
#     if not a_ngrams or not b_ngrams:
#         return 0.0
#     inter = a_ngrams & b_ngrams
#     # ratio relative to min length (與前述規則一致)
#     return len(inter) / max(1, min(len(a_ngrams), len(b_ngrams)))

# def cosine_distance(a_emb, b_emb):
#     # a_emb, b_emb 都為 numpy 或 tensor
#     return 1 - util.cos_sim(a_emb, b_emb).item()

# # ---------- 驗證函式 ----------
# def validate_caption(post: str, synopsis: str, sentiment: str,
#                      min_len=80, max_len=200,
#                      max_ngram_overlap=0.8, min_cos_dist=0.1):
#     # 長度（以字元計算）
#     L = len(post)
#     if not (min_len <= L <= max_len):
#         return False, f"len={L}"

#     # ngram 與 synopsis/sentiment
#     ov1 = ngram_overlap(post, synopsis, n=3)
#     ov2 = ngram_overlap(post, sentiment, n=3)
#     if ov1 > max_ngram_overlap:
#         return False, f"ngram_syn={ov1:.3f}"
#     if ov2 > max_ngram_overlap:
#         return False, f"ngram_sen={ov2:.3f}"

#     # cosine distance（用 synopsis+sentiment 的合併）
#     ref = (str(synopsis) + " " + str(sentiment)).strip()
#     if len(ref) == 0:
#         return False, "no_ref_text"
#     emb_post = embedder.encode(post, convert_to_tensor=True)
#     emb_ref = embedder.encode(ref, convert_to_tensor=True)
#     dist = cosine_distance(emb_post, emb_ref)
#     if dist < min_cos_dist:
#         return False, f"cos_dist={dist:.3f}"

#     return True, f"ok (ng:{ov1:.3f}/{ov2:.3f} cos:{dist:.3f})"

# # ---------- 生成貼文（含清理 + 檢核 + retry） ----------
# def generate_post_with_validation(synopsis, sentiment, phq8_score, dep_desc,
#                                   max_retries=4):
#     """
#     逐次嘗試不同溫度 / top_p 組合生成，並在每次生成後清理與驗證，
#     若驗證通過則回傳 clean_text；否則嘗試下一組參數，最後回傳解析失敗字串。
#     """
#     # prompt：非常嚴格，**只輸出一段繁體中文貼文**，不得有 JSON / code / 英文 / emoji
#     prompt = f"""
# 你是一位社群使用者，會用自然口語在 IG 上發文。

# 請根據以下資訊，生成一則**單段**完整貼文（切勿包含任何範例、JSON、程式碼或額外說明）：
# - 摘要：{synopsis}
# - 情緒分析：{sentiment}
# - PHQ-8 分數：{phq8_score}（{dep_desc}）

# 請務必遵守以下規則：
# 1) 只輸出**一則純文字**貼文（不要任何標題、範例、JSON、程式碼區塊或額外補充）。
# 2) 全部使用 **繁體中文**（嚴禁使用英文或夾雜英文單字）。
# 3) **不得**輸出代碼區塊 、大括號、或任何 JSON 樣式內容。
# 4) 長度請控制在 80–200 字，以自然、口語化語氣表達，像在 IG 抒發心情。
# 5) 不要包含 emoji（如 😔）、表情符號、或非中文符號。
# 6) 若違反上述任何規則，請回傳空字串。

# 只需直接輸出那則貼文，不要任何其他文字。
# """

#     # 嘗試不同參數組合（從比較穩定到較隨機）
#     param_grid = [
#         {"temperature": 0.2, "top_p": 0.9},
#         {"temperature": 0.3, "top_p": 0.9},
#         {"temperature": 0.5, "top_p": 0.95},
#         {"temperature": 0.7, "top_p": 0.95},
#     ]

#     last_raw = ""
#     for i in range(min(max_retries, len(param_grid))):
#         params = param_grid[i]
#         out = generator(prompt,
#                         max_new_tokens=300,
#                         do_sample=True,
#                         temperature=params["temperature"],
#                         top_p=params["top_p"],
#                         return_full_text=False)[0]["generated_text"]
#         last_raw = out
#         # 清理
#         cleaned = cleanup_generated_text(out)

#         # 若清理結果為空，跳下一次
#         if not cleaned:
#             continue

#         # 檢核
#         ok, reason = validate_caption(cleaned, synopsis, sentiment)
#         if ok:
#             return cleaned, {"status": "ok", "attempt": i+1, "params": params, "reason": reason}
#         # 若不合格，繼續 retry
#     # 全部失敗 -> 回傳解析失敗（包含最後一次 raw 以便除錯）
#     fallback = "【解析失敗】 " + cleanup_generated_text(last_raw)
#     return fallback, {"status": "fail", "attempt": min(max_retries, len(param_grid)), "last_raw": last_raw}

# # ---------- 主流程：逐列生成並立即檢核 ----------
# results = []
# stats = {"ok": 0, "fail": 0}
# new_post_id = 410

# for _, row in tqdm(df.iterrows(), total=len(df), desc="Stage3 - generate & validate"):
#     syn = str(row.get("synthetic_synopsis", "")).strip()
#     sen = str(row.get("synthetic_sentiment", "")).strip()
#     pid = int(row["post_id"])
#     phq = int(row["phq8_score"])
#     dep = row["dep_desc"]

#     # 若 Stage2 有解析失敗，跳過
#     if syn.startswith("【解析失敗】") or sen.startswith("【解析失敗】") or not syn or not sen:
#         results.append({
#             "post_id": new_post_id,
#             "caption": "【解析失敗】stage2缺材料",
#             "phq8_score": phq,
#             "dep_desc": dep,
#             "valid": False,
#             "note": "stage2_fail_or_missing"
#         })
#         new_post_id += 1
#         stats["fail"] += 1
#         continue

#     cap, meta = generate_post_with_validation(syn, sen, phq, dep, max_retries=4)
#     valid_flag = meta["status"] == "ok"

#     results.append({
#         "post_id": new_post_id,
#         "caption": cap,
#         "phq8_score": phq,
#         "dep_desc": dep,
#         "valid": valid_flag,
#         "meta": str(meta)
#     })
#     if valid_flag:
#         stats["ok"] += 1
#     else:
#         stats["fail"] += 1

#     new_post_id += 1

# # 輸出
# out_df = pd.DataFrame(results)
# out_df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
# out_df.to_csv(OUT_CHECKED, index=False, encoding="utf-8-sig")

# print("Stage3 完成")
# print("合格：", stats["ok"], "不合格：", stats["fail"])
# print("輸出檔案：", OUT_PATH)


#####  加入檢核標準：
##### 字數檢查：必須在 80–200 字之間。
##### n-gram 重疊率：與摘要/情緒分析比較，重疊率 ≤ 0.8。
##### cosine distance：與原始摘要/情緒分析的 embedding 距離 ≥ 0.1